# Agent Laboratory + memória entre projetos · Grupo 1 · LLM 2026/2

**QP1.** Guardar lições de projetos anteriores faz o agente se sair melhor nos próximos?
**QP2.** Importa de onde vem a lição: do revisor automático ou de um humano?

| Versão | Memória |
|---|---|
| **A** | sem memória (Agent Laboratory original) |
| **B** | lições tiradas do feedback do **revisor LLM** |
| **C** | lições tiradas do **nosso feedback** |

**Protocolo:** 10 temas (os 5 do artigo + 5 novos). Os 5 de **treino** rodam uma vez, e os mesmos relatórios
recebem os dois tipos de feedback, gerando as memórias B e C. Os 5 de **teste** rodam nas versões A, B e C, com a
memória congelada. Medimos a qualidade (avaliação humana cega, 1–5), as falhas que se repetem por etapa, e o custo e o tempo.

**Antes de começar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`.

**A cada sessão nova do Colab**, rode a Parte 0 inteira e depois a etapa em que vocês estão. Tudo fica salvo no
Google Drive, e as etapas continuam de onde pararam.

| Parte | Quando (cronograma) |
|---|---|
| 0 · Preparação | toda sessão |
| 1 · Treino, memória B, testes A e B | até 23/10 |
| 2 · Feedback humano, memória C, teste C | até 13/11 |
| 3 · Avaliação cega e análise | até 13/11 |

# Parte 0 · Preparação (toda sessão)
## 0.1 Configuração
Para o primeiro teste: `MODO_RAPIDO = True`, `SEEDS = 1`. Para o resultado final: `MODO_RAPIDO = False`.

In [ ]:
MODELO = "qwen2.5:7b"        # o MESMO modelo em todas as versões (alternativa: "qwen2.5:14b", melhor e mais lento)
CONTEXTO = 16384             # janela de contexto do Ollama, em tokens
SEEDS = 1                    # repetições do protocolo inteiro
MODO_RAPIDO = True           # True: menos passos por fase (bom para validar o fluxo)
PASTA_DRIVE = "/content/drive/MyDrive/agent_lab_projeto"

NUM_PAPERS, MAX_STEPS, MLE_STEPS = (3, 10, 2) if MODO_RAPIDO else (5, 20, 3)
BASE_URL = "http://127.0.0.1:11434/v1"
ARGS = (f"--model {MODELO} --base_url {BASE_URL} --seeds {SEEDS} --num_papers {NUM_PAPERS} "
        f"--max_steps {MAX_STEPS} --mle_steps {MLE_STEPS} --out {PASTA_DRIVE}")
print(ARGS)

## 0.2 Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
os.makedirs(PASTA_DRIVE, exist_ok=True)
print("Resultados em:", PASTA_DRIVE)

## 0.3 GPU
Deve aparecer uma Tesla T4 (ou melhor).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0.4 Dependências e Ollama

In [ ]:
!pip -q install arxiv pypdf datasets huggingface_hub
!apt-get -qq install -y zstd lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 0.5 Servidor Ollama e modelo
O aviso `systemd is not running` da célula anterior é normal: esta célula sobe o servidor manualmente.

In [ ]:
import subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH=str(CONTEXTO), OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"),
                               stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); print("Servidor Ollama no ar."); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama não subiu. Veja /content/ollama.log")
!ollama pull {MODELO}
os.environ["OPENAI_API_KEY"] = "ollama"

## 0.6 Código do projeto
Cada célula grava um arquivo em `/content/agent_lab`. Não precisa editar.

In [ ]:
os.makedirs("/content/agent_lab/agent_lab_memory", exist_ok=True)
%cd /content/agent_lab

In [ ]:
%%writefile /content/agent_lab/agent_lab_memory/__init__.py
"""Agent Laboratory com memória de longo prazo."""
from .laboratory import AgentLaboratory
from .llm import LLM, MockLLM
from .memory import LongTermMemory, Embedder

__all__ = ["AgentLaboratory", "LLM", "MockLLM", "LongTermMemory", "Embedder"]

In [ ]:
%%writefile /content/agent_lab/agent_lab_memory/llm.py
"""
Camada de acesso a LLMs.

- LLM: wrapper para APIs compatíveis com OpenAI (OpenAI, vLLM, Ollama, OpenRouter...)
  ou para a API da Anthropic. Contabiliza tokens e chamadas (útil para comparar
  custo com e sem memória).
- MockLLM: respostas determinísticas, só para testar o pipeline sem gastar API.
"""
from __future__ import annotations

import json
import os
import random
import re
import time
from dataclasses import dataclass, asdict


@dataclass
class Usage:
    calls: int = 0
    prompt_tokens: int = 0
    completion_tokens: int = 0

    def to_dict(self) -> dict:
        return asdict(self)


class LLM:
    def __init__(self, model: str = "gpt-4o-mini", provider: str = "openai",
                 temperature: float = 0.8, max_tokens: int = 4096,
                 base_url: str | None = None, api_key: str | None = None, seed: int | None = None):
        self.model, self.provider = model, provider
        self.seed = seed  # reduz a variância entre as versões A/B/C (OpenAI e Ollama aceitam 'seed')
        self.temperature, self.max_tokens = temperature, max_tokens
        self.usage = Usage()
        if provider == "openai":
            from openai import OpenAI
            self._client = OpenAI(base_url=base_url, api_key=api_key or os.getenv("OPENAI_API_KEY"))
        elif provider == "anthropic":
            import anthropic
            self._client = anthropic.Anthropic(api_key=api_key or os.getenv("ANTHROPIC_API_KEY"))
        else:
            raise ValueError(f"provider desconhecido: {provider}")

    def query(self, system: str, prompt: str, temperature: float | None = None) -> str:
        temp = self.temperature if temperature is None else temperature
        last_err = None
        for attempt in range(4):
            try:
                if self.provider == "openai":
                    extra = {"seed": self.seed} if self.seed is not None else {}
                    r = self._client.chat.completions.create(
                        model=self.model, temperature=temp, max_tokens=self.max_tokens, **extra,
                        messages=[{"role": "system", "content": system},
                                  {"role": "user", "content": prompt}])
                    text = r.choices[0].message.content or ""
                    if r.usage:
                        self.usage.prompt_tokens += r.usage.prompt_tokens
                        self.usage.completion_tokens += r.usage.completion_tokens
                else:
                    r = self._client.messages.create(
                        model=self.model, temperature=temp, max_tokens=self.max_tokens,
                        system=system, messages=[{"role": "user", "content": prompt}])
                    text = "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
                    self.usage.prompt_tokens += r.usage.input_tokens
                    self.usage.completion_tokens += r.usage.output_tokens
                self.usage.calls += 1
                log_call(self, system, prompt, text)
                return text
            except Exception as e:  # rate limit, timeout...
                last_err = e
                time.sleep(2 ** attempt)
        raise RuntimeError(f"Falha ao consultar o LLM: {last_err}")


class MockLLM:
    """LLM falso: devolve comandos válidos para cada fase. Serve só para testar o
    fluxo (memória, persistência, métricas) sem custo. Os 'resultados' são aleatórios."""

    def __init__(self, seed: int = 0):
        self.usage = Usage()
        self.rng = random.Random(seed)

    def query(self, system: str, prompt: str, temperature: float | None = None) -> str:
        self.usage.calls += 1
        self.usage.prompt_tokens += (len(system) + len(prompt)) // 4
        text = self._answer(system, prompt)
        self.usage.completion_tokens += len(text) // 4
        log_call(self, system, prompt, text)
        return text

    def _answer(self, system: str, prompt: str) -> str:
        cur = re.findall(r"Current Step #\d+, Phase: (\w+)", prompt)  # fase ATUAL (não a do histórico)
        s = system + "\n" + (f"Phase: {cur[-1]}" if cur else prompt)
        bonus = 0.0  # mock neutro: não simula ganho com memória (isso é o que o experimento real mede)
        if "expert reward model" in system:
            return f"```SCORE\n{min(1.0, self.rng.uniform(0.4, 0.8) + bonus):.2f}\n```"
        if "reviewing a paper" in system:
            o = min(10, self.rng.randint(3, 6) + (1 if bonus else 0))
            return "THOUGHT: ok\nREVIEW JSON:\n```json\n" + json.dumps({
                "Summary": "mock", "Strengths": ["clear"], "Weaknesses": ["small dataset", "no significance test"],
                "Originality": 2, "Quality": 2, "Clarity": 3, "Significance": 2, "Soundness": 2,
                "Presentation": 3, "Contribution": 2, "Overall": o, "Confidence": 3,
                "Decision": "Reject" if o < 6 else "Accept"}) + "\n```"
        if "extract reusable lessons" in system:
            src = "human" if "human researchers" in prompt else "llm"
            return "```json\n" + json.dumps([
                {"problem": f"[{src}] No statistical significance test for the accuracy gap.",
                 "how_to_avoid": "Report a bootstrap confidence interval for every comparison.",
                 "phase": "running_experiments", "importance": 0.8},
                {"problem": f"[{src}] Dataset too large for the timeout.",
                 "how_to_avoid": "Load a small subset (e.g. split='test[:500]').",
                 "phase": "data_preparation", "importance": 0.6}]) + "\n```"
        if "code repair tool" in system:
            return "```python\nprint('accuracy: 0.70')\n```"
        if "Reflect" in prompt or "reflect" in prompt[:200]:
            return "Use a stronger baseline and print per-class accuracy."
        if "Phase: literature_review" in s:
            if "Paper summaries" in prompt or self.rng.random() < 0.5:
                return f"```ADD_PAPER\n2401.0{self.rng.randint(1000, 9999)}\nMock paper relevant to the topic.\n```"
            return "```SUMMARY\nlanguage model bias benchmark\n```"
        if "Phase: plan_formulation" in s:
            if "postdoctoral" in system:
                return "```PLAN\n1. Load a small HF dataset. 2. Compare baseline vs. method. 3. Report accuracy.\n```"
            return "```DIALOGUE\nI propose comparing a baseline with our method on a small dataset.\n```"
        if "Phase: data_preparation" in s:
            if "PhD student" in system:
                return ("```SUBMIT_CODE\nimport random\nrandom.seed(0)\n"
                        "data = [(random.random(), random.randint(0, 1)) for _ in range(200)]\n"
                        "print('dataset size', len(data))\n```")
            return "```DIALOGUE\nI will generate a small toy dataset.\n```"
        if "Phase: running_experiments" in s or "ML engineer and you will be writing the code" in s:
            acc = min(0.99, self.rng.uniform(0.55, 0.8) + bonus)
            return (f"```REPLACE\nimport random\nacc = {acc:.3f}\n"
                    "print('Results show baseline vs method accuracy')\n"
                    f"print('accuracy:', acc)\n```")
        if "Phase: results_interpretation" in s:
            if "postdoctoral" in system:
                return "```INTERPRETATION\nThe method improved accuracy by ~5 points over the baseline.\n```"
            return "```DIALOGUE\nThe method seems better than the baseline.\n```"
        if "report writer" in system or "section" in prompt.lower():
            return "```REPLACE\nMock section text with numbers: accuracy 0.70.\n```"
        if "decide" in prompt.lower():
            return "```FINISH\n```"
        return "```DIALOGUE\nok\n```"


def log_call(llm, system: str, prompt: str, response: str) -> None:
    """Grava cada chamada em JSONL (se llm.log_path estiver definido) para diagnóstico."""
    path = getattr(llm, "log_path", None)
    if not path:
        return
    try:
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps({"t": round(time.time(), 1), "system": system[:400], "prompt_tail": prompt[-600:],
                                "response": response}, ensure_ascii=False) + "\n")
    except OSError:
        pass


_THINK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)


def extract_command(text: str, command: str) -> str | None:
    """Extrai o conteúdo de um bloco ```COMMAND ... ``` (formato usado no artigo).

    Modelos menores raramente seguem o formato à risca, então aceitamos variações:
      ```PLAN\n...```            (formato canônico)
      ```plan\n...```            (minúsculas)
      ```PLAN: texto```          (tudo numa linha, com ou sem ':')
      ```text\nPLAN\n...```      (tag em minúsculas antes; comando em MAIÚSCULAS)
      ```PLAN\n...  (sem fechar) (resposta cortada: lê até o fim)
      PLAN\n...                   (sem crases, no início de uma linha, em MAIÚSCULAS)
    Blocos <think>...</think> de modelos de raciocínio são ignorados.
    """
    text = _THINK.sub("", text or "")
    cmd = re.escape(command)
    tail = r"(?![A-Za-z0-9_])[ \t]*:?[ \t]*([^\n]*?)[ \t]*(?:\n(.*?))?(?:```|\Z)"
    patterns = [re.compile(r"```[ \t]*" + cmd + tail, re.DOTALL | re.IGNORECASE)]
    if command.lower() != "python":
        patterns.append(re.compile(r"```[ \t]*[a-z]*[ \t]*\n[ \t]*" + cmd + tail, re.DOTALL))
        # comando "solto", sem crases, no início de uma linha (ex.: "PLAN\n..." ou "**PLAN:** ...");
        # só em MAIÚSCULAS, para não confundir com a palavra comum em prosa
        patterns.append(re.compile(r"(?m)^[ \t]*(?:\*\*)?" + cmd + r"(?:\*\*)?" + tail, re.DOTALL))
    for pat in patterns:
        m = pat.search(text)
        if m:
            header = (m.group(1) or "").strip().lstrip("*").strip()  # "**PLAN:** texto"
            body = (m.group(2) or "").strip()
            return "\n".join(x for x in (header, body) if x)
    return None

In [ ]:
%%writefile /content/agent_lab/agent_lab_memory/memory.py
"""
Memória de longo prazo para o Agent Laboratory.

O agente original só tem memória de curto prazo (o histórico do prompt, com
decaimento) e descarta tudo ao final de cada projeto. Aqui adicionamos uma
memória persistente entre projetos, inspirada em Generative Agents (Park et al.,
2023), Reflexion (Shinn et al., 2024) e ExpeL (Zhao et al., 2024).

Tipos de memória (campo `kind`):
  - episodic   : resumo do que aconteceu em cada fase de um projeto (plano, resultados, notas)
  - insight    : lições generalizáveis extraídas por reflexão no fim do projeto
  - error      : erros de código encontrados pelo mle-solver e como foram corrigidos
  - literature : artigos do arXiv já lidos e julgados relevantes (cache semântico)
  - feedback   : notas do humano no modo co-pilot (preferências do pesquisador)

Recuperação (retrieve): pontuação = w_sim * similaridade + w_imp * importância + w_rec * recência
  - similaridade : cosseno entre embeddings (sentence-transformers se instalado,
                   senão um bag-of-words com TF-IDF em Python puro)
  - importância  : 0..1, dada na escrita (ex.: lições de reviews ruins pesam mais)
  - recência     : decay ** (projetos decorridos desde o último acesso)
"""
from __future__ import annotations

import json
import math
import os
import re
import time
import uuid
from collections import Counter
from dataclasses import dataclass, field, asdict

from .llm import extract_command

KINDS = ("episodic", "insight", "error", "literature", "feedback")

# Quais tipos de memória cada fase consulta.
PHASE_KINDS = {
    "literature_review": ["literature", "insight", "feedback"],
    "plan_formulation": ["insight", "episodic", "feedback"],
    "data_preparation": ["insight", "error", "feedback"],
    "running_experiments": ["insight", "error", "episodic", "feedback"],
    "results_interpretation": ["insight", "episodic"],
    "report_writing": ["insight", "feedback"],
    "report_refinement": ["insight"],
}


@dataclass
class MemoryItem:
    content: str
    kind: str
    topic: str
    phase: str | None = None
    importance: float = 0.5
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:10])
    created_run: int = 0
    last_access_run: int = 0
    access_count: int = 0
    created_at: float = field(default_factory=time.time)
    metadata: dict = field(default_factory=dict)


# ----------------------------------------------------------------------------- embeddings
_TOKEN = re.compile(r"[a-zA-ZÀ-ÿ0-9_]+")


def _tokens(text: str) -> list[str]:
    return [t.lower() for t in _TOKEN.findall(text) if len(t) > 2]


class Embedder:
    """Usa sentence-transformers se disponível; caso contrário, TF-IDF esparso."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", use_dense: bool = True):
        self.dense = None
        if use_dense:
            try:
                from sentence_transformers import SentenceTransformer
                self.dense = SentenceTransformer(model_name)
            except Exception:
                self.dense = None

    def similarities(self, query: str, docs: list[str]) -> list[float]:
        if not docs:
            return []
        if self.dense is not None:
            vecs = self.dense.encode([query] + docs, normalize_embeddings=True)
            q = vecs[0]
            return [float(max(0.0, q @ v)) for v in vecs[1:]]
        # TF-IDF em Python puro
        tok_docs = [Counter(_tokens(d)) for d in docs]
        q_tok = Counter(_tokens(query))
        n = len(docs) + 1
        df = Counter()
        for c in tok_docs + [q_tok]:
            df.update(c.keys())
        idf = {t: math.log(n / df[t]) + 1.0 for t in df}

        def vec(c):
            v = {t: f * idf[t] for t, f in c.items()}
            norm = math.sqrt(sum(x * x for x in v.values())) or 1.0
            return {t: x / norm for t, x in v.items()}

        qv = vec(q_tok)
        return [sum(qv.get(t, 0.0) * x for t, x in vec(c).items()) for c in tok_docs]


# ----------------------------------------------------------------------------- store
class LongTermMemory:
    def __init__(self, path: str = "memory_store.json", embedder: Embedder | None = None,
                 w_sim: float = 0.6, w_imp: float = 0.25, w_rec: float = 0.15,
                 recency_decay: float = 0.9, dedup_threshold: float = 0.9,
                 max_items_per_kind: int = 200, read_only: bool = False):
        self.path = path
        self.read_only = read_only  # True na fase de TESTE: a memória é consultada mas nunca alterada
        self.embedder = embedder or Embedder()
        self.w_sim, self.w_imp, self.w_rec = w_sim, w_imp, w_rec
        self.recency_decay = recency_decay
        self.dedup_threshold = dedup_threshold
        self.max_items_per_kind = max_items_per_kind
        self.items: list[MemoryItem] = []
        self.run_counter = 0  # quantos projetos já passaram por esta memória
        self.retrieval_log: list[dict] = []  # para análise: o que foi recuperado em cada fase
        self.load()

    # --- persistência
    def load(self):
        if os.path.exists(self.path):
            with open(self.path, encoding="utf-8") as f:
                data = json.load(f)
            self.run_counter = data.get("run_counter", 0)
            self.items = [MemoryItem(**d) for d in data.get("items", [])]

    def save(self):
        if self.read_only:
            return
        os.makedirs(os.path.dirname(os.path.abspath(self.path)), exist_ok=True)
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump({"run_counter": self.run_counter,
                       "items": [asdict(i) for i in self.items]}, f, ensure_ascii=False, indent=2)

    def start_run(self):
        if not self.read_only:
            self.run_counter += 1
        self.retrieval_log = []

    # --- escrita
    def add(self, content: str, kind: str, topic: str, phase: str | None = None,
            importance: float = 0.5, **metadata) -> MemoryItem | None:
        assert kind in KINDS, kind
        if self.read_only:
            return None
        content = content.strip()
        same_kind = [i for i in self.items if i.kind == kind]
        if same_kind:  # deduplicação: memória quase igual → reforça em vez de duplicar
            sims = self.embedder.similarities(content, [i.content for i in same_kind])
            best = max(range(len(sims)), key=sims.__getitem__)
            if sims[best] >= self.dedup_threshold:
                it = same_kind[best]
                it.importance = min(1.0, max(it.importance, importance) + 0.05)
                it.last_access_run = self.run_counter
                it.metadata.setdefault("reinforced", 0)
                it.metadata["reinforced"] += 1
                return it
        it = MemoryItem(content=content, kind=kind, topic=topic, phase=phase,
                        importance=float(max(0.0, min(1.0, importance))),
                        created_run=self.run_counter, last_access_run=self.run_counter,
                        metadata=metadata)
        self.items.append(it)
        self._evict(kind)
        return it

    def _evict(self, kind: str):
        same = [i for i in self.items if i.kind == kind]
        if len(same) <= self.max_items_per_kind:
            return
        same.sort(key=lambda i: i.importance * self._recency(i))
        drop = {i.id for i in same[: len(same) - self.max_items_per_kind]}
        self.items = [i for i in self.items if i.id not in drop]

    # --- leitura
    def _recency(self, it: MemoryItem) -> float:
        return self.recency_decay ** max(0, self.run_counter - it.last_access_run)

    def retrieve(self, query: str, kinds: list[str] | None = None, phase: str | None = None,
                 k: int = 6, min_sim: float = 0.05) -> list[MemoryItem]:
        pool = [i for i in self.items if (kinds is None or i.kind in kinds)]
        if not pool:
            return []
        sims = self.embedder.similarities(query, [f"{i.topic}. {i.content}" for i in pool])
        scored = []
        for it, sim in zip(pool, sims):
            # lições são gerais por definição: as da PRÓPRIA fase sempre concorrem, mesmo com pouca
            # semelhança textual com o tema (senão o filtro descartaria justamente o que é útil)
            same_phase = phase is not None and it.phase == phase
            if sim < min_sim and it.kind != "feedback" and not same_phase:
                continue
            phase_bonus = 0.1 if (phase and it.phase == phase) else 0.0
            score = self.w_sim * sim + self.w_imp * it.importance + self.w_rec * self._recency(it) + phase_bonus
            scored.append((score, it))
        scored.sort(key=lambda x: x[0], reverse=True)
        chosen = [it for _, it in scored[:k]]
        for it in chosen:
            it.access_count += 1
            it.last_access_run = self.run_counter
        self.retrieval_log.append({"phase": phase, "query": query[:200], "ids": [i.id for i in chosen]})
        return chosen

    def context_for_phase(self, phase: str, topic: str, extra_query: str = "", k: int = 6) -> str:
        items = self.retrieve(f"{topic}\n{extra_query}", kinds=PHASE_KINDS.get(phase), phase=phase, k=k)
        return format_memories(items)

    def stats(self) -> dict:
        c = Counter(i.kind for i in self.items)
        return {"runs": self.run_counter, "total": len(self.items), **{k: c.get(k, 0) for k in KINDS}}


def format_memories(items: list[MemoryItem]) -> str:
    if not items:
        return ""
    labels = {"episodic": "projeto anterior", "insight": "lição", "error": "erro conhecido",
              "literature": "artigo já lido", "feedback": "preferência do pesquisador"}
    lines = ["=== MEMÓRIA DE LONGO PRAZO (experiência de projetos anteriores) ===",
             "Use estas informações se forem úteis; ignore-as se não se aplicarem ao tópico atual."]
    for it in items:
        lines.append(f"- [{labels[it.kind]} | tópico: {it.topic[:60]}] {it.content}")
    return "\n".join(lines) + "\n"


# ----------------------------------------------------------------------------- lições
PHASES_LESSON = ["literature_review", "plan_formulation", "data_preparation", "running_experiments",
                 "results_interpretation", "report_writing"]

LESSON_SYSTEM = (
    "You extract reusable lessons from feedback on a finished automated ML research project, so that FUTURE "
    "projects on OTHER topics avoid the same problems. Each lesson states what went wrong and how to avoid it. "
    "Base the lessons ONLY on the feedback and the project facts given; never invent problems that are not "
    "mentioned. Lessons must be general (not specific to this topic), concrete and actionable, written in English "
    "even if the feedback is in another language. Return ONLY a JSON list inside ```json ... ``` where each element is "
    '{"problem": str, "how_to_avoid": str, "phase": one of ' + json.dumps(PHASES_LESSON) + ', "importance": float 0..1}. '
    "Higher importance for problems that hurt the result most. At most 4 lessons. "
    "(Task id: extract reusable lessons.)"
)

SOURCE_LABEL = {"llm": "automated LLM reviewer (NeurIPS-style review)",
                "human": "human researchers who read the report"}


def project_facts(result: dict, max_chars: int = 2500) -> str:
    """Fatos objetivos do projeto (iguais para as condições B e C): resultado das fases, falhas, erros."""
    tr = result.get("trace", {})
    fails = result.get("failures", [])
    fail_txt = "; ".join(f"{f['fase']}:{f['categoria']}" for f in fails) or "none detected"
    errs = "\n".join(e.get("text", "")[-300:] for e in tr.get("errors", [])[:4])
    txt = (f"Phase success: {json.dumps(result.get('metrics', {}).get('phase_success', {}))}\n"
           f"Automatically detected failures: {fail_txt}\n"
           f"Plan (excerpt): {tr.get('plan', '')[:600]}\n"
           f"Final experiment output (excerpt): {tr.get('exp_results', '')[-600:]}\n"
           f"Code errors (sample):\n{errs}")
    return txt[:max_chars]


def extract_lessons(llm, topic: str, facts: str, feedback: str, source: str) -> list[dict]:
    """Transforma feedback (do revisor LLM ou de humanos) em lições 'problema -> como evitar'.
    O MESMO extrator é usado para B e C; só muda o texto do feedback (QP2)."""
    prompt = (f"Research topic: {topic}\n\nPROJECT FACTS:\n{facts}\n\n"
              f"FEEDBACK (source: {SOURCE_LABEL.get(source, source)}):\n{feedback}\n\nNow extract the lessons.")
    raw = llm.query(LESSON_SYSTEM, prompt, temperature=0.2)
    body = extract_command(raw, "json") or raw
    try:
        data = json.loads(body)
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]", body, re.DOTALL)
        try:
            data = json.loads(m.group(0)) if m else []
        except json.JSONDecodeError:
            data = []
    lessons = []
    for d in data if isinstance(data, list) else []:
        if not isinstance(d, dict):
            continue
        prob = d.get("problem") or d.get("lesson")
        if not prob:
            continue
        lessons.append({"problem": str(prob).strip(), "how_to_avoid": str(d.get("how_to_avoid", "")).strip(),
                        "phase": d.get("phase") if d.get("phase") in PHASES_LESSON else None,
                        "importance": float(d.get("importance", 0.5) or 0.5)})
    return lessons[:4]


def store_lessons(memory: LongTermMemory, lessons: list[dict], topic: str, source: str, project: str) -> int:
    n = 0
    for les in lessons:
        content = f"Problem: {les['problem']} | How to avoid: {les['how_to_avoid']}".strip(" |")
        if memory.add(content, "insight", topic, phase=les["phase"], importance=les["importance"],
                      source=source, project=project) is not None:
            n += 1
    return n


def format_llm_review(review: dict) -> str:
    """Feedback do revisor automático em texto (entrada da condição B)."""
    if not review:
        return "The automated reviewer produced no review (the project did not reach the report stage)."
    keys = ["Overall", "Soundness", "Presentation", "Contribution", "Quality", "Clarity", "Significance"]
    scores = ", ".join(f"{k}={review[k]}" for k in keys if k in review)
    parts = [f"Scores: {scores}. Decision: {review.get('Decision', '?')}"]
    for k in ("Weaknesses", "Questions", "Limitations"):
        v = review.get(k)
        if v:
            parts.append(f"{k}: " + ("; ".join(map(str, v)) if isinstance(v, list) else str(v)))
    return "\n".join(parts)


def consolidate_run(memory: LongTermMemory, llm, result: dict, project: str = "") -> int:
    """Modo 'online' (opcional): ao fim de cada projeto, lições do revisor LLM entram direto na memória."""
    if memory.read_only:
        return 0
    lessons = extract_lessons(llm, result["topic"], project_facts(result), format_llm_review(result.get("review")), "llm")
    n = store_lessons(memory, lessons, result["topic"], "llm", project)
    memory.save()
    return n

In [ ]:
%%writefile /content/agent_lab/agent_lab_memory/failures.py
"""
Detecção automática de falhas por etapa (métrica "falhas que se repetem" da proposta).

Cada projeto gera uma lista de eventos {"fase": ..., "categoria": ...}. Na análise, uma falha
"se repete" quando a mesma (fase, categoria) já tinha acontecido nos projetos de TREINO,
ou seja, a memória teve a chance de ensinar o agente a evitá-la e não ensinou.

Categorias (inspiradas nas falhas da Seção 5.2 do artigo):
  fase_falhou          a fase não terminou com sucesso
  formato_invalido     o modelo respondeu sem um comando válido
  busca_em_loop        a revisão bibliográfica gastou todos os passos
  dataset_inexistente  tentou carregar um dataset que não existe / exige script
  dependencia_ausente  ModuleNotFoundError / ImportError
  timeout              o código excedeu o tempo limite
  erro_de_sintaxe      SyntaxError / IndentationError
  falta_de_memoria     CUDA out of memory / MemoryError
  erro_de_execucao     KeyError, ValueError, TypeError, IndexError, AttributeError...
  outro_erro           qualquer outro erro de código
  acuracia_zero        o experimento reportou 0% de acurácia
  sem_relatorio        o projeto terminou sem relatório
"""
from __future__ import annotations

import re

_ERR_RULES = [
    ("dataset_inexistente", r"DatasetNotFoundError|doesn't exist on the Hub|Couldn't find|FileNotFoundError|"
                            r"trust_remote_code|Dataset scripts are no longer supported|EmptyDatasetError"),
    ("dependencia_ausente", r"ModuleNotFoundError|ImportError|No module named"),
    ("timeout", r"Timeout|TimeoutExpired"),
    ("erro_de_sintaxe", r"SyntaxError|IndentationError"),
    ("falta_de_memoria", r"out of memory|OutOfMemoryError|MemoryError"),
    ("erro_de_execucao", r"KeyError|ValueError|TypeError|IndexError|AttributeError|NameError|ZeroDivisionError|RuntimeError"),
]
_ACC = re.compile(r"acc(?:uracy)?\w*[^0-9\n]{0,12}?(\d+(?:\.\d+)?)\s*(%?)", re.IGNORECASE)


def classify_error(text: str) -> str:
    for cat, pat in _ERR_RULES:
        if re.search(pat, text or "", re.IGNORECASE):
            return cat
    return "outro_erro"


def has_zero_accuracy(results: str) -> bool:
    vals = [float(m.group(1)) for m in _ACC.finditer(results or "")]
    return bool(vals) and all(v == 0.0 for v in vals)


def detect_failures(result: dict, max_steps: int) -> list[dict]:
    m = result.get("metrics", {})
    tr = result.get("trace", {})
    ev: list[dict] = []
    for phase, ok in m.get("phase_success", {}).items():
        if not ok:
            ev.append({"fase": phase, "categoria": "fase_falhou"})
    for phase, n in m.get("invalid_commands", {}).items():
        if n:
            ev.append({"fase": phase, "categoria": "formato_invalido", "n": n})
    if m.get("phase_steps", {}).get("literature_review", 0) >= max_steps:
        ev.append({"fase": "literature_review", "categoria": "busca_em_loop"})
    for e in tr.get("errors", []):
        ev.append({"fase": e.get("phase", "?"), "categoria": classify_error(e.get("text", ""))})
    if tr.get("exp_results") and has_zero_accuracy(tr["exp_results"]):
        ev.append({"fase": "running_experiments", "categoria": "acuracia_zero"})
    if not result.get("review"):
        ev.append({"fase": "report_writing", "categoria": "sem_relatorio"})
    return ev

In [ ]:
%%writefile /content/agent_lab/agent_lab_memory/tools.py
"""Ferramentas: busca no arXiv, busca de datasets no HuggingFace e execução de código."""
from __future__ import annotations

import os
import re
import subprocess
import sys
import tempfile


class ArxivSearch:
    def __init__(self, max_results: int = 20):
        self.max_results = max_results
        try:
            import arxiv
            self._client = arxiv.Client(page_size=max_results, delay_seconds=3, num_retries=3)
            self._arxiv = arxiv
        except ImportError:
            self._client = None

    def summaries(self, query: str) -> str:
        if self._client is None:
            return "arXiv indisponível (pip install arxiv)."
        search = self._arxiv.Search(query=query[:300], max_results=self.max_results,
                                    sort_by=self._arxiv.SortCriterion.Relevance)
        out = []
        try:
            for r in self._client.results(search):
                out.append(f"Title: {r.title}\narXiv ID: {r.get_short_id()}\n"
                           f"Published: {r.published.date()}\nSummary: {r.summary[:800]}\n")
        except Exception as e:
            return f"Erro no arXiv: {e}"
        return "\n".join(out) or "Nenhum resultado."

    def full_text(self, arxiv_id: str, max_chars: int = 20000) -> str:
        if self._client is None:
            return "arXiv indisponível."
        try:
            from pypdf import PdfReader
            paper = next(self._client.results(self._arxiv.Search(id_list=[arxiv_id.strip()])))
            with tempfile.TemporaryDirectory() as d:
                pdf = paper.download_pdf(dirpath=d)
                text = "\n".join((p.extract_text() or "") for p in PdfReader(pdf).pages)
            return text[:max_chars]
        except Exception as e:
            return f"Não foi possível obter o texto completo: {e}"


def search_hf_datasets(query: str, limit: int = 10) -> str:
    try:
        from huggingface_hub import list_datasets
        res = list_datasets(search=query, limit=limit, sort="downloads")
        lines = [f"- {d.id} (downloads: {getattr(d, 'downloads', '?')})" for d in res]
        return "\n".join(lines) or "Nenhum dataset encontrado."
    except Exception as e:
        return f"Busca no HuggingFace indisponível: {e}"


_FORBIDDEN = [r"\bexit\s*\(\s*\)", r"\bquit\s*\(\s*\)", r"sys\.exit\s*\([^)]*\)", r"os\._exit\s*\([^)]*\)"]


def sanitize_code(code: str) -> str:
    """O artigo relata que o mle-solver gerava exit() e matava o processo; removemos isso."""
    for pat in _FORBIDDEN:
        code = re.sub(pat, "pass", code)
    return code


def execute_code(code: str, timeout: int = 600, workdir: str | None = None, max_chars: int = 4000) -> tuple[bool, str]:
    """Roda o código num subprocesso separado (isolamento mínimo).
    ATENÇÃO: código gerado por LLM deve rodar em container/VM descartável."""
    code = sanitize_code(code)
    workdir = workdir or tempfile.mkdtemp(prefix="agentlab_")
    path = os.path.join(workdir, "run.py")
    with open(path, "w", encoding="utf-8") as f:
        f.write(code)
    try:
        p = subprocess.run([sys.executable, path], cwd=workdir, capture_output=True,
                           text=True, timeout=timeout)
        out = (p.stdout or "") + (("\n[stderr]\n" + p.stderr) if p.stderr else "")
        ok = p.returncode == 0
    except subprocess.TimeoutExpired:
        ok, out = False, f"[ERRO] Timeout de {timeout}s excedido."
    if len(out) > max_chars:  # evita estourar o limite de tokens (falha citada no artigo)
        out = out[: max_chars // 2] + "\n...[saída truncada]...\n" + out[-max_chars // 2:]
    return ok, out

In [ ]:
%%writefile /content/agent_lab/agent_lab_memory/agents.py
"""
Agentes (PhD, Postdoc, ML Engineer, Professor) no estilo do Agent Laboratory.

A estrutura do prompt segue o Apêndice B.1 do artigo (system = papel + instruções da fase
+ comandos; user = contexto + histórico + passo + tópico + feedback + notas). A única
diferença estrutural é o bloco `memory_context`, inserido antes do histórico.
Os textos dos prompts são versões condensadas/adaptadas dos do Apêndice B.
"""
from __future__ import annotations

ROLE = {
    "phd": "a computer science PhD student at a top university",
    "postdoc": "a computer science postdoctoral researcher at a top university",
    "mle": "a machine learning engineer working at a top university",
    "professor": "a computer science professor at a top university",
}

CMD_FORMAT = ("Use exactly ONE command per turn, wrapped in triple backticks: ```COMMAND\\n<content>\\n``` "
              "where COMMAND is the actual command name.")

PHASE_TASK = {
    ("phd", "literature_review"):
        "Perform a literature review for the research topic and add relevant papers. "
        "You can search arXiv for paper summaries and read full texts. Do not overuse SUMMARY: add papers.",
    ("phd", "plan_formulation"):
        "You talk with a postdoc to design a SIMPLE experiment for the topic, building on the literature. "
        "Specify models, datasets (HuggingFace) and exact experimental steps.",
    ("postdoc", "plan_formulation"):
        "You direct a PhD student to design a SIMPLE experiment for the topic, building on the literature. "
        "When the plan is good, submit it. Do not talk for too long.",
    ("phd", "data_preparation"):
        "You direct an ML engineer to write SIMPLE code that prepares the data for the plan. "
        "When the code is ready, submit it. Use ONLY a dataset id from the verified list in the context "
        "(or a torchvision/sklearn built-in dataset); never invent dataset names. Keep the data small.",
    ("mle", "data_preparation"):
        "Write SIMPLE data preparation code for the plan, guided by the PhD student. You can run code and search HF "
        "datasets. Use ONLY dataset ids from the verified list (or torchvision/sklearn); never invent names.",
    ("phd", "results_interpretation"):
        "Discuss with the postdoc how to interpret the experimental results. Use the real numbers, metrics and significance.",
    ("postdoc", "results_interpretation"):
        "Guide the PhD student to an accurate interpretation of the results (with numbers and significance). "
        "Submit it in reasonable time.",
}

PHASE_COMMANDS = {
    ("phd", "literature_review"):
        "```SUMMARY\\n<search query>\\n``` returns top arXiv summaries.\n"
        "```FULL_TEXT\\n<arXiv id>\\n``` returns the full paper text.\n"
        "```ADD_PAPER\\n<arXiv id>\\n<summary of the paper and its results>\\n``` adds a paper to the review.",
    ("phd", "plan_formulation"): "```DIALOGUE\\n<message>\\n```",
    ("postdoc", "plan_formulation"): "```DIALOGUE\\n<message>\\n```  or  ```PLAN\\n<final plan>\\n```",
    ("phd", "data_preparation"): "```DIALOGUE\\n<message>\\n```  or  ```SUBMIT_CODE\\n<final python code>\\n```",
    ("mle", "data_preparation"):
        "```python\\n<code to run>\\n```  or  ```SEARCH_HF\\n<query>\\n```  or  ```DIALOGUE\\n<message>\\n```",
    ("phd", "results_interpretation"): "```DIALOGUE\\n<message>\\n```",
    ("postdoc", "results_interpretation"): "```DIALOGUE\\n<message>\\n```  or  ```INTERPRETATION\\n<interpretation>\\n```",
}


class Agent:
    def __init__(self, kind: str, llm, max_history: int = 10, notes: dict | None = None):
        self.kind, self.llm = kind, llm
        self.max_history = max_history
        self.history: list[str] = []      # memória de CURTO prazo (como no original)
        self.prev_comm = ""
        self.notes = notes or {}           # notas do humano por fase (task notes do artigo)

    def system_prompt(self, phase: str) -> str:
        return (f"You are {ROLE[self.kind]}.\n"
                f"Task instructions: {PHASE_TASK.get((self.kind, phase), '')}\n"
                f"Commands available:\n{PHASE_COMMANDS.get((self.kind, phase), '')}\n{CMD_FORMAT}")

    def step(self, topic: str, phase: str, step: int, max_steps: int, feedback: str = "",
             context: str = "", memory_context: str = "") -> str:
        complete = ("You must finish this task and submit as soon as possible!"
                    if step >= 0.7 * max_steps else "")
        hist = "\n".join(self.history[-self.max_history:])
        prompt = (f"{context}\n{memory_context}\nHistory:\n{hist}\n"
                  f"Current Step #{step}, Phase: {phase}\n{complete}\n"
                  f"[Objective] Your goal is to perform research on the following topic: {topic}\n"
                  f"Feedback: {feedback}\nNotes: {self.notes.get(phase, '')}\n"
                  f"Your previous command was: {self.prev_comm[:300]}. Make sure your new output is different.\n"
                  "Please produce a single command below:")
        resp = self.llm.query(self.system_prompt(phase), prompt)
        self.history.append(f"Step #{step}, Phase: {phase}, Feedback: {feedback[:800]}, Your response: {resp[:800]}")
        self.prev_comm = resp
        return resp

    def reset(self):
        self.history, self.prev_comm = [], ""

In [ ]:
%%writefile /content/agent_lab/agent_lab_memory/solvers.py
"""
mle-solver, paper-solver e revisor automático (versões compactas das Seções 3.2 e 3.3).
Pontos de integração com a memória estão marcados com  # [MEMÓRIA].
"""
from __future__ import annotations

import json
import random
import re

from .llm import extract_command
from .tools import execute_code

# ============================================================================ mle-solver
MLE_ROLE = ("You are an expert machine learning engineer working at a top university to write code "
            "to solve machine learning research challenges.")
MLE_PHASE = ("You are an ML engineer and you will be writing the code for a research project. Produce SIMPLE code "
             "that runs every experiment in the plan and prints the results. The dataset code is always prepended "
             "to your code, so do not rewrite it. Do not write a main function. Use only pytorch or scikit-learn "
             "(no tensorflow/keras). Before printing each result, print what it is meant to show. "
             "Your method must not get 0% accuracy; if it does, something is wrong. Keep runtime short.\n"
             "If the experiment needs a language model, call the local model exactly like this "
             "(and evaluate at most 50 examples):\n"
             "import os\nfrom openai import OpenAI\n"
             "client = OpenAI(base_url=os.environ.get('AGENTLAB_LLM_BASE_URL'), api_key=os.environ.get('OPENAI_API_KEY', 'x'))\n"
             "r = client.chat.completions.create(model=os.environ['AGENTLAB_LLM_MODEL'], "
             "messages=[{'role': 'user', 'content': prompt}], max_tokens=64, temperature=0)\n"
             "answer = r.choices[0].message.content")
MLE_CMDS = ("Commands (ONE per turn):\n"
            "```REPLACE\\n<entire new python file>\\n``` rewrites all code (use for big changes).\n"
            "```EDIT N M\\n<new lines>\\n``` replaces lines N..M (inclusive) of the current code.\n"
            "Code is tested before being accepted.")
SCORER_SYS = ("You are a professor agent serving as an expert reward model: read a research plan, research code and "
              "its output, and score from 0 to 1 how well the code follows the plan and produces proper results. "
              "Answer exactly as ```SCORE\\n<float>\\n```")
REPAIR_SYS = ("You are an automated code repair tool. Fix the code so the error does not happen again, changing as "
              "little as possible. Return the full code inside ```python ... ```.")


class MLESolver:
    def __init__(self, llm, plan: str, dataset_code: str, insights: str = "", notes: str = "",
                 steps: int = 3, top_k: int = 2, n_repair: int = 2, timeout: int = 600,
                 memory=None, topic: str = ""):
        self.llm, self.plan, self.dataset_code = llm, plan, dataset_code
        self.insights, self.notes = insights, notes
        self.steps, self.top_k, self.n_repair, self.timeout = steps, top_k, n_repair, timeout
        self.memory, self.topic = memory, topic
        self.top: list[tuple[float, list[str], str]] = []  # (score, linhas, saída)
        self.reflection = ""
        self.errors: list[str] = []          # usado na consolidação da memória
        self.stats = {"exec_attempts": 0, "exec_errors": 0, "repairs_ok": 0}

    # ---- memória: erros parecidos e lições relevantes
    def _memory_block(self, query: str) -> str:
        if self.memory is None:
            return ""
        return self.memory.context_for_phase("running_experiments", self.topic, extra_query=query, k=6)  # [MEMÓRIA]

    def _system(self, code_lines: list[str], last_output: str) -> str:
        code = "\n".join(f"{i} |{l}" for i, l in enumerate(code_lines)) or "(empty)"
        return (f"{MLE_ROLE}\nTask instructions: {MLE_PHASE}\n"
                f"Insights from the literature review:\n{self.insights[:3000]}\n"
                f"{self._memory_block(self.plan[:500] + last_output[-500:])}"
                f"Reflection on previous attempts:\n{self.reflection}\n"
                f"Notes: {self.notes}\nResearch plan:\n{self.plan}\n"
                f"Dataset code (prepended automatically):\n{self.dataset_code[:3000]}\n{MLE_CMDS}\n"
                f"Current code:\n{code}")

    def _run(self, lines: list[str]) -> tuple[bool, str]:
        self.stats["exec_attempts"] += 1
        ok, out = execute_code(self.dataset_code + "\n" + "\n".join(lines), timeout=self.timeout)
        if not ok:
            self.stats["exec_errors"] += 1
        return ok, out

    def _repair(self, lines: list[str], error: str) -> tuple[bool, list[str], str]:
        for _ in range(self.n_repair):
            mem = self._memory_block(error[-800:])  # [MEMÓRIA] o reparo consulta erros já vistos
            resp = self.llm.query(REPAIR_SYS + "\n" + mem, f"Error:\n{error[-2000:]}\n\nCode:\n" + "\n".join(lines),
                                  temperature=0.8)
            fixed = extract_command(resp, "python")
            if fixed is None:
                continue
            new_lines = fixed.split("\n")
            ok, out = self._run(new_lines)
            if ok:
                self.stats["repairs_ok"] += 1
                if self.memory is not None:  # [MEMÓRIA] grava erro + solução
                    self.memory.add(f"Erro: {error.strip().splitlines()[-1][:300]} -> corrigido por reparo automático.",
                                    "error", self.topic, phase="running_experiments", importance=0.5)
                return True, new_lines, out
            error = out
        return False, lines, error

    def _score(self, lines: list[str], output: str) -> float:
        resp = self.llm.query(SCORER_SYS, f"Plan:\n{self.plan}\n\nCode:\n" + "\n".join(lines) +
                              f"\n\nOutput:\n{output[-3000:]}", temperature=0.6)
        body = extract_command(resp, "SCORE") or resp
        m = re.search(r"[01](?:\.\d+)?", body)
        return float(m.group(0)) if m else 0.0

    def _apply(self, resp: str, lines: list[str]) -> list[str] | None:
        rep = extract_command(resp, "REPLACE")
        if rep is not None:
            return rep.split("\n")
        ed = extract_command(resp, "EDIT")
        if ed is not None:
            header, _, body = ed.partition("\n")
            nums = re.findall(r"\d+", header)
            if len(nums) >= 2:
                n, m = int(nums[0]), int(nums[1])
                return lines[:n] + body.split("\n") + lines[m + 1:]
        return None

    def solve(self) -> tuple[str, str, float]:
        lines: list[str] = []
        last_out = ""
        # passo inicial: gerar código do zero (até 5 tentativas)
        for _ in range(5):
            resp = self.llm.query(self._system([], ""), "Use ```REPLACE to create the initial code now.", temperature=1.0)
            new = self._apply(resp, [])
            if new is None:
                continue
            ok, out = self._run(new)
            if not ok:
                self.errors.append(out[-500:])
                ok, new, out = self._repair(new, out)
            if ok:
                self.top.append((self._score(new, out), new, out))
                break
        for step in range(self.steps):
            if not self.top:
                break
            _, base, last_out = random.choice(self.top)  # amostragem do pool de melhores programas
            resp = self.llm.query(self._system(base, last_out), f"Step {step}: improve the code with one command.",
                                  temperature=1.0)
            new = self._apply(resp, base)
            if new is None:
                continue
            ok, out = self._run(new)
            if not ok:
                self.errors.append(out[-500:])
                ok, new, out = self._repair(new, out)
            if ok:
                sc = self._score(new, out)
                self.top.append((sc, new, out))
                self.top = sorted(self.top, key=lambda t: t[0], reverse=True)[: self.top_k]
                msg = f"Code ran. Score {sc:.2f}. Output:\n{out[-1500:]}\nReflect on how to improve this result."
            else:
                msg = f"Code failed with error:\n{out[-1500:]}\nReflect on why and how to fix it."
            self.reflection = self.llm.query("You are an ML engineer. Reflect briefly and concretely.", msg, temperature=0.8)
        if not self.top:
            return "", "Nenhum código executou com sucesso.", 0.0
        best = max(self.top, key=lambda t: t[0])
        return "\n".join(best[1]), best[2], best[0]


# ============================================================================ revisor
REVIEW_SYS = (
    "You are an AI researcher who is reviewing a paper that was submitted to a prestigious ML venue. "
    "Be critical and cautious. Respond with 'THOUGHT:' (your reasoning) and then 'REVIEW JSON:' with a ```json``` block "
    "containing: Summary, Strengths (list), Weaknesses (list), Originality, Quality, Clarity, Significance (1-4), "
    "Questions, Limitations, Ethical Concerns (bool), Soundness, Presentation, Contribution (1-4), "
    "Overall (1-10, NeurIPS scale: 6 weak accept, 5 borderline accept, 4 borderline reject, 3 reject), "
    "Confidence (1-5), Decision (Accept or Reject). Deduct points if any core section is missing."
)


def review_paper(llm, plan: str, latex: str) -> dict:
    resp = llm.query(REVIEW_SYS, f"Research plan:\n{plan}\n\nPaper:\n{latex[:20000]}", temperature=0.5)
    body = extract_command(resp, "json") or resp
    try:
        return json.loads(body)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", body, re.DOTALL)
        try:
            return json.loads(m.group(0)) if m else {}
        except json.JSONDecodeError:
            return {}


# ============================================================================ paper-solver
SECTIONS = ["Abstract", "Introduction", "Background", "Related Work", "Methods",
            "Experimental Setup", "Results", "Discussion"]
TIPS = {
    "Abstract": "One paragraph: what, why it matters, why hard, contribution, how verified.",
    "Related Work": "Compare and contrast with alternative approaches; do not just describe them.",
    "Methods": "Precise methodology and equations.",
    "Experimental Setup": "Dataset, metrics, hyperparameters. Do not invent hardware details.",
    "Results": "ONLY results that were actually run. Numbers, baselines, limitations. Never hallucinate results.",
}
PAPER_SYS = ("You are a computer science PhD student acting as a report writer: you summarize research that was "
             "actually performed into an academic report. Writing style: objective. Write LaTeX body text only "
             "(no packages, no \\section commands). Return it as ```REPLACE\\n<text>\\n```.")


class PaperSolver:
    def __init__(self, llm, topic: str, plan: str, lit_review: str, exp_code: str, exp_results: str,
                 interpretation: str, notes: str = "", refine_steps: int = 2, memory=None):
        self.llm, self.topic, self.plan = llm, topic, plan
        self.lit, self.code, self.results, self.interp = lit_review, exp_code, exp_results, interpretation
        self.notes, self.refine_steps, self.memory = notes, refine_steps, memory
        self.sections: dict[str, str] = {}

    def _context(self) -> str:
        mem = self.memory.context_for_phase("report_writing", self.topic, k=5) if self.memory else ""  # [MEMÓRIA]
        return (f"{mem}Topic: {self.topic}\nNotes: {self.notes}\nLiterature review:\n{self.lit[:3000]}\n"
                f"Plan:\n{self.plan[:2000]}\nCode:\n{self.code[:3000]}\nResults:\n{self.results[:2000]}\n"
                f"Interpretation:\n{self.interp[:1500]}\n")

    def latex(self) -> str:
        body = "\n".join(f"\\section{{{s}}}\n{self.sections.get(s, '')}" for s in SECTIONS)
        return ("\\documentclass{article}\n\\usepackage{amsmath}\n\\title{Research Report: " + self.topic +
                "}\n\\author{Agent Laboratory}\n\\begin{document}\n\\maketitle\n" + body + "\n\\end{document}")

    def markdown(self) -> str:
        """Versão legível do relatório (para leitura e avaliação humana)."""
        body = "\n\n".join(f"## {s}\n\n{self.sections.get(s, '')}" for s in SECTIONS)
        return f"# Research Report: {self.topic}\n\n{body}\n"

    def write_section(self, s: str, extra: str = "") -> None:
        resp = self.llm.query(PAPER_SYS, f"{self._context()}\nCurrent paper sections so far: "
                              f"{', '.join(self.sections)}\n{extra}\nWrite the section: {s}. Tips: {TIPS.get(s, '')}",
                              temperature=0.8)
        text = extract_command(resp, "REPLACE")
        self.sections[s] = text if text is not None else resp

    def solve(self) -> tuple[str, dict]:
        for s in SECTIONS:
            self.write_section(s)
        best_tex = self.latex()
        best_rev = review_paper(self.llm, self.plan, best_tex)
        for _ in range(self.refine_steps):  # edição guiada pela revisão; mantém a melhor versão
            weak = "; ".join(best_rev.get("Weaknesses", [])[:4])
            target = random.choice(["Introduction", "Methods", "Results", "Discussion"])
            old = self.sections[target]
            self.write_section(target, extra=f"Reviewer weaknesses to address: {weak}\nPrevious version:\n{old[:3000]}")
            tex = self.latex()
            rev = review_paper(self.llm, self.plan, tex)
            if rev.get("Overall", 0) >= best_rev.get("Overall", 0):
                best_tex, best_rev = tex, rev
            else:
                self.sections[target] = old
        return best_tex, best_rev

In [ ]:
%%writefile /content/agent_lab/agent_lab_memory/laboratory.py
"""
Orquestrador do fluxo (Figura 2 do artigo) com memória de longo prazo opcional.

Fases: literature_review -> plan_formulation -> data_preparation -> running_experiments
       -> results_interpretation -> report_writing -> report_refinement

Onde a memória entra (# [MEMÓRIA]):
  LEITURA : no início de cada fase, recupera as lições relevantes ao tópico/fase e injeta no prompt.
  ESCRITA : só acontece se a memória NÃO for somente-leitura. No protocolo do projeto (projeto.py)
            a memória é construída offline a partir do feedback (LLM ou humano) e fica congelada
            durante os testes, então nada é escrito aqui.
Com memory=None o comportamento é o do Agent Laboratory original (versão A).
"""
from __future__ import annotations

import json
import os
import re
import time

from .agents import Agent
from .llm import extract_command
from .failures import detect_failures
from .memory import LongTermMemory, consolidate_run
from .solvers import MLESolver, PaperSolver
from .tools import ArxivSearch, execute_code, search_hf_datasets

PHASES = ["literature_review", "plan_formulation", "data_preparation", "running_experiments",
          "results_interpretation", "report_writing", "report_refinement"]


class AgentLaboratory:
    def __init__(self, topic: str, llm, memory: LongTermMemory | None = None, copilot: bool = False,
                 notes: dict | None = None, num_papers: int = 5, max_steps: int = 20,
                 mle_steps: int = 3, paper_steps: int = 2, max_revisits: int = 1,
                 exp_timeout: int = 600, out_dir: str = "runs/run", verbose: bool = True):
        self.topic, self.llm, self.memory = topic, llm, memory
        self.copilot, self.notes = copilot, notes or {}
        self.num_papers, self.max_steps = num_papers, max_steps
        self.mle_steps, self.paper_steps, self.max_revisits = mle_steps, paper_steps, max_revisits
        self.exp_timeout, self.out_dir, self.verbose = exp_timeout, out_dir, verbose
        self.phd, self.postdoc = Agent("phd", llm, notes=self.notes), Agent("postdoc", llm, notes=self.notes)
        self.mle = Agent("mle", llm, notes=self.notes)
        self.arxiv = ArxivSearch()
        os.makedirs(out_dir, exist_ok=True)
        self.llm.log_path = os.path.join(out_dir, "llm_calls.jsonl")  # respostas brutas, para diagnóstico
        self.last_response: dict[str, str] = {}
        # estado do projeto
        self.lit_review: list[dict] = []
        self.plan = self.dataset_code = self.exp_code = self.exp_results = ""
        self.interpretation = self.report = self.report_md = ""
        self.review: dict = {}
        self.mle_score = 0.0
        self.human_feedback: list[str] = []
        self.errors: list[dict] = []   # {"phase": ..., "text": ...}
        self.metrics: dict = {"phase_steps": {}, "phase_success": {}, "phase_time": {}, "invalid_commands": {}}

    # ------------------------------------------------------------------ utilidades
    def log(self, *a):
        if self.verbose:
            print("[AgentLab]", *a, flush=True)

    def mem(self, phase: str, extra: str = "") -> str:
        if self.memory is None:
            return ""
        return self.memory.context_for_phase(phase, self.topic, extra_query=extra)  # [MEMÓRIA] leitura

    def invalid(self, phase: str, resp: str, example: str) -> str:
        """Conta o erro de formato e devolve um feedback que mostra o formato correto."""
        self.metrics["invalid_commands"][phase] = self.metrics["invalid_commands"].get(phase, 0) + 1
        return ("Your last output did not contain a valid command, so nothing happened. "
                f"Reply with exactly one fenced command, for example:\n{example}")

    def lit_str(self) -> str:
        return "\n".join(f"[{p['id']}] {p['summary']}" for p in self.lit_review)

    def checkpoint(self, phase: str, output: str) -> str:
        """Modo co-pilot: o humano revisa a saída da fase; nota vira memória de 'feedback'."""
        if not self.copilot:
            return ""
        print(f"\n===== Saída da fase {phase} =====\n{output[:3000]}\n")
        fb = input("Enter = seguir | ou escreva uma nota para refazer a fase: ").strip()
        if fb:
            self.human_feedback.append(f"{phase}: {fb}")
            if self.memory is not None:  # [MEMÓRIA] preferência do pesquisador, alta importância
                self.memory.add(fb, "feedback", self.topic, phase=phase, importance=0.9)
        return fb

    # ------------------------------------------------------------------ fases
    def literature_review(self) -> bool:
        phase, feedback = "literature_review", ""
        known = self.mem(phase)  # [MEMÓRIA] artigos já lidos em projetos parecidos + lições
        for step in range(self.max_steps):
            resp = self.phd.step(self.topic, phase, step, self.max_steps, feedback,
                                 context=f"Papers added so far ({len(self.lit_review)}/{self.num_papers}):\n{self.lit_str()}",
                                 memory_context=known)
            if (q := extract_command(resp, "SUMMARY")) is not None:
                feedback = f"Paper summaries for '{q}':\n{self.arxiv.summaries(q)[:6000]}"
            elif (pid := extract_command(resp, "FULL_TEXT")) is not None:
                feedback = f"Full text:\n{self.arxiv.full_text(pid)[:8000]}"
            elif (add := extract_command(resp, "ADD_PAPER")) is not None:
                pid, _, summ = add.partition("\n")
                self.lit_review.append({"id": pid.strip(), "summary": summ.strip()})
                feedback = f"Paper {pid.strip()} added."
                if self.memory is not None:  # [MEMÓRIA] cache semântico de literatura
                    self.memory.add(f"arXiv {pid.strip()}: {summ.strip()[:500]}", "literature", self.topic,
                                    phase=phase, importance=0.4, arxiv_id=pid.strip())
            else:
                feedback = self.invalid(phase, resp, "```SUMMARY\nlanguage models differential diagnosis MedQA\n```\n"
                                        "or\n```ADD_PAPER\n2401.01234\nOne-paragraph summary of the paper and its results.\n```")
            self.last_response[phase] = resp
            if len(self.lit_review) >= self.num_papers:
                self.metrics["phase_steps"][phase] = step + 1
                return True
        self.metrics["phase_steps"][phase] = self.max_steps
        return len(self.lit_review) > 0

    def _dialogue(self, phase: str, a: Agent, b: Agent, submit_cmd: str, context: str) -> str | None:
        """Diálogo genérico: 'a' propõe, 'b' dirige e submete com submit_cmd."""
        memory_context = self.mem(phase)  # [MEMÓRIA]
        fb_a = fb_b = reminder = ""
        for step in range(self.max_steps):
            ra = a.step(self.topic, phase, step, self.max_steps, fb_a, context, memory_context)
            fb_b = extract_command(ra, "DIALOGUE") or ra[:1500]
            rb = b.step(self.topic, phase, step, self.max_steps, fb_b, context + reminder, memory_context)
            self.last_response[phase] = rb
            if (sub := extract_command(rb, submit_cmd)) is not None and sub:
                self.metrics["phase_steps"][phase] = step + 1
                return sub
            fb_a = extract_command(rb, "DIALOGUE")
            if fb_a is None:
                fb_a = rb[:1500]
                self.invalid(phase, rb, f"```DIALOGUE\nyour message\n```  or  ```{submit_cmd}\nfinal text\n```")
            if step + 1 >= self.max_steps // 2:  # lembrete explícito para quem precisa submeter
                reminder = (f"\nREMINDER: you are running out of steps. Submit now with "
                            f"```{submit_cmd}\n<final text>\n```\n")
        self.metrics["phase_steps"][phase] = self.max_steps
        return None

    def suggest_datasets(self) -> str:
        """Busca datasets REAIS no HuggingFace antes da preparação de dados.
        Modelos pequenos inventam nomes (ex.: load_dataset('medqa')); com a lista verificada, escolhem um que existe."""
        sys_p = ("Return 3 short HuggingFace dataset search queries (1 to 3 words each) for the experiment below, "
                 "one per line, no numbering, no explanation.")
        raw = self.llm.query(sys_p, f"Topic: {self.topic}\nPlan:\n{self.plan[:1500]}", temperature=0.3)
        queries = []
        for line in raw.splitlines():
            q = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line).strip().strip("`\"'").strip()
            if q and not q.isupper() and len(q) <= 40 and q.lower() not in ("text", "plaintext"):
                queries.append(q)
        found: list[str] = []
        for q in queries[:3]:
            for line in search_hf_datasets(q, limit=5).splitlines():
                if line.startswith("- ") and line not in found:
                    found.append(line)
        self.metrics["hf_suggestions"] = {"queries": queries[:3], "found": len(found)}
        lst = "\n".join(found[:12]) if found else "(no results; use a torchvision or sklearn built-in dataset)"
        return ("VERIFIED DATASETS that exist on HuggingFace (use one of these EXACT ids with load_dataset):\n"
                f"{lst}\nAlternatively use a torchvision (e.g. CIFAR10) or sklearn built-in dataset. "
                "Load only a small subset, e.g. split='test[:500]'.\n")

    def plan_formulation(self) -> bool:
        plan = self._dialogue("plan_formulation", self.phd, self.postdoc, "PLAN",
                              f"Current Literature Review:\n{self.lit_str()}")
        self.plan = plan or ""
        return bool(plan)

    def data_preparation(self) -> bool:
        phase = "data_preparation"
        memory_context = self.mem(phase, self.plan[:500])  # [MEMÓRIA]
        context = (f"Current Literature Review:\n{self.lit_str()}\nCurrent Plan:\n{self.plan}\n"
                   f"{self.suggest_datasets()}")
        fb_mle = fb_phd = ""
        for step in range(self.max_steps):
            rm = self.mle.step(self.topic, phase, step, self.max_steps, fb_mle, context, memory_context)
            if (code := extract_command(rm, "python")) is not None:
                ok, out = execute_code(code, timeout=300)
                fb_mle = f"Code output:\n{out}"
                fb_phd = f"The ML engineer ran code. Output:\n{out[:1500]}"
            elif (q := extract_command(rm, "SEARCH_HF")) is not None:
                fb_mle = f"HF datasets:\n{search_hf_datasets(q)}"
                fb_phd = f"The ML engineer searched HF for '{q}'."
            else:
                fb_phd = extract_command(rm, "DIALOGUE") or rm[:1500]
            rp = self.phd.step(self.topic, phase, step, self.max_steps, fb_phd, context, memory_context)
            self.last_response[phase] = rp
            if (code := extract_command(rp, "SUBMIT_CODE")) is not None and code:
                ok, out = execute_code(code, timeout=300)
                if ok:
                    self.dataset_code = code
                    self.metrics["phase_steps"][phase] = step + 1
                    return True
                self.errors.append({"phase": phase, "text": out[-500:]})
                if self.memory is not None:  # [MEMÓRIA] erro na preparação de dados
                    self.memory.add(f"Erro ao preparar dados: {out.strip().splitlines()[-1][:300] if out.strip() else 'sem saída'}",
                                    "error", self.topic, phase=phase, importance=0.5)
                fb_mle = fb_phd = (f"Submitted code failed:\n{out[-1500:]}\nIf the dataset was not found or needs a "
                                   "loading script, choose ANOTHER exact ID from the verified list (or a torchvision/"
                                   "sklearn built-in dataset). Never invent dataset names.")
            else:
                fb_mle = extract_command(rp, "DIALOGUE")
                if fb_mle is None:
                    fb_mle = rp[:1500]
                    self.invalid(phase, rp, "```DIALOGUE\nmessage\n```  or  ```SUBMIT_CODE\nfinal python code\n```")
        self.metrics["phase_steps"][phase] = self.max_steps
        return False

    def running_experiments(self) -> bool:
        solver = MLESolver(self.llm, self.plan, self.dataset_code, insights=self.lit_str(),
                           notes=self.notes.get("running_experiments", ""), steps=self.mle_steps,
                           timeout=self.exp_timeout, memory=self.memory, topic=self.topic)
        self.exp_code, self.exp_results, self.mle_score = solver.solve()
        self.errors += [{"phase": "running_experiments", "text": e} for e in solver.errors]
        self.metrics["mle_solver"] = solver.stats
        return bool(self.exp_code)

    def results_interpretation(self) -> bool:
        ctx = (f"Current Literature Review:\n{self.lit_str()}\nCurrent Plan:\n{self.plan}\n"
               f"Current Dataset code:\n{self.dataset_code[:2000]}\nCurrent Experiment code:\n{self.exp_code[:3000]}\n"
               f"Current Results:\n{self.exp_results[:3000]}")
        interp = self._dialogue("results_interpretation", self.phd, self.postdoc, "INTERPRETATION", ctx)
        self.interpretation = interp or ""
        return bool(interp)

    def report_writing(self) -> bool:
        ps = PaperSolver(self.llm, self.topic, self.plan, self.lit_str(), self.exp_code, self.exp_results,
                         self.interpretation, notes=self.notes.get("report_writing", ""),
                         refine_steps=self.paper_steps, memory=self.memory)
        self.report, self.review = ps.solve()
        self.report_md = ps.markdown()
        return bool(self.report)

    def report_refinement(self) -> str:
        """PhD decide: FINISH ou voltar para um subtarefa (plan/experiment/interpretation)."""
        sys = ("You are a computer science PhD student who submitted a paper. Given the reviews, decide whether to "
               "finish or revisit an earlier stage. Reply ```FINISH\\n``` or ```REVISIT\\n<plan_formulation|"
               "running_experiments|results_interpretation>\\n```.")
        mem = self.mem("report_refinement")  # [MEMÓRIA]
        resp = self.llm.query(sys, f"{mem}Review:\n{json.dumps(self.review)[:3000]}\nPlease decide now.")
        rev = extract_command(resp, "REVISIT")
        return rev.strip() if rev else "FINISH"

    # ------------------------------------------------------------------ execução
    def _run_phase(self, phase: str) -> bool:
        # memória de curto prazo é POR FASE: sem isso o modelo imita comandos da fase anterior
        for ag in (self.phd, self.postdoc, self.mle):
            ag.reset()
        t0 = time.time()
        ok = getattr(self, phase)()
        self.metrics["phase_time"][phase] = round(time.time() - t0, 1)
        self.metrics["phase_success"][phase] = bool(ok)
        inv = self.metrics["invalid_commands"].get(phase, 0)
        self.log(f"{phase}: {'ok' if ok else 'FALHOU'} ({self.metrics['phase_steps'].get(phase, '-')} passos, "
                 f"{inv} comandos inválidos, {self.metrics['phase_time'][phase]}s)")
        if not ok and phase in self.last_response:
            self.log("   última resposta do modelo: " + self.last_response[phase][:300].replace("\n", " | "))
        errs = [e["text"] for e in self.errors if e.get("phase") == phase]
        if not ok and errs:  # o erro de execução é o que explica a falha, não só a resposta do modelo
            linhas = [l for l in errs[-1].strip().splitlines() if l.strip()]
            self.log(f"   erros de execução nesta fase: {len(errs)}; último: " + " | ".join(linhas[-3:])[:400])
        return ok

    def run(self) -> dict:
        t0 = time.time()
        if self.memory is not None:
            self.memory.start_run()
        order = PHASES[:-1]
        i = 0
        while i < len(order):
            phase = order[i]
            ok = self._run_phase(phase)
            out = {"literature_review": self.lit_str(), "plan_formulation": self.plan,
                   "data_preparation": self.dataset_code, "running_experiments": self.exp_results,
                   "results_interpretation": self.interpretation, "report_writing": self.report}[phase]
            fb = self.checkpoint(phase, out)
            if fb:  # co-pilot pediu para refazer
                if phase == "literature_review":
                    self.lit_review = []
                self.notes[phase] = (self.notes.get(phase, "") + "\n" + fb).strip()
                for ag in (self.phd, self.postdoc, self.mle):
                    ag.reset()
                continue
            if not ok and phase in ("data_preparation", "running_experiments"):
                self.log("Fase crítica falhou; encerrando o projeto.")
                break
            i += 1
            if i == len(order) and self.max_revisits > 0:
                target = self.report_refinement()
                self.metrics.setdefault("revisits", []).append(target)
                if target in order:
                    self.max_revisits -= 1
                    i = order.index(target)
        result = self.finalize(self.summary(time.time() - t0))
        if self.memory is not None:
            if not self.memory.read_only:  # [MEMÓRIA] modo online opcional: grava lições do revisor LLM
                consolidate_run(self.memory, self.llm, result, project=self.out_dir)
            result["memory_stats"] = self.memory.stats()
            result["memory_retrievals"] = self.memory.retrieval_log
        self.save(result)
        return result

    def summary(self, elapsed: float) -> dict:
        ps = self.metrics["phase_success"]
        return {
            "topic": self.topic, "with_memory": self.memory is not None,
            "review": self.review, "overall": self.review.get("Overall"),
            "mle_score": self.mle_score, "n_code_errors": len(self.errors),
            "workflow_success": round(sum(ps.values()) / max(1, len(ps)), 3),
            "usage": self.llm.usage.to_dict(), "elapsed_s": round(elapsed, 1), "metrics": self.metrics,
            "trace": {"plan": self.plan, "exp_results": self.exp_results[-3000:],
                      "interpretation": self.interpretation, "errors": self.errors[:30],
                      "n_papers": len(self.lit_review), "human_feedback": self.human_feedback},
        } | {"failures": []}

    def finalize(self, result: dict) -> dict:
        result["failures"] = detect_failures(result, self.max_steps)
        return result

    def feedback_summary(self, result: dict) -> str:
        """Tudo o que um humano precisa para dar feedback sobre o projeto (condição C), num só arquivo."""
        fails = "\n".join(f"- {f['fase']}: {f['categoria']}" for f in result.get("failures", [])) or "- nenhuma detectada"
        ok = "\n".join(f"- {k}: {'ok' if v else 'FALHOU'}" for k, v in self.metrics["phase_success"].items())
        return (f"# Projeto: {self.topic}\n\n## Etapas\n{ok}\n\n## Falhas detectadas automaticamente\n{fails}\n\n"
                f"## Artigos da revisão ({len(self.lit_review)})\n{self.lit_str() or '(nenhum)'}\n\n"
                f"## Plano\n{self.plan or '(sem plano)'}\n\n## Código de preparação de dados\n```python\n"
                f"{self.dataset_code[:3000]}\n```\n\n## Código do experimento\n```python\n{self.exp_code[:4000]}\n```\n\n"
                f"## Saída do experimento\n```\n{self.exp_results[-2500:]}\n```\n\n## Interpretação\n"
                f"{self.interpretation or '(sem interpretação)'}\n\n## Relatório final\n\n{self.report_md or '(sem relatório)'}\n")

    def save(self, result: dict):
        os.makedirs(self.out_dir, exist_ok=True)
        files = {"report.tex": self.report, "report.md": self.report_md, "resumo_para_feedback.md": self.feedback_summary(result),
                 "plan.txt": self.plan, "dataset_code.py": self.dataset_code,
                 "experiment_code.py": self.exp_code, "results.txt": self.exp_results,
                 "interpretation.txt": self.interpretation, "literature.txt": self.lit_str()}
        for name, content in files.items():
            with open(os.path.join(self.out_dir, name), "w", encoding="utf-8") as f:
                f.write(content or "")
        with open(os.path.join(self.out_dir, "result.json"), "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

In [ ]:
%%writefile /content/agent_lab/projeto.py
"""
Protocolo experimental do projeto (Grupo 1 · LLM 2026/2)

  QP1. Guardar lições de projetos anteriores faz o agente se sair melhor nos próximos?
  QP2. Importa de onde vem a lição: do revisor automático ou de um humano?

Versões do agente
  A  sem memória (Agent Laboratory original)
  B  memória com lições tiradas do REVISOR LLM
  C  memória com lições tiradas do FEEDBACK HUMANO

Etapas (cada uma é um subcomando; todas podem ser retomadas se a sessão cair):

  python projeto.py treino            # roda os 5 temas de treino UMA vez (sem memória)
  python projeto.py memoria --fonte llm      # constrói a memória B a partir das revisões LLM
  python projeto.py teste --versao A         # roda os 5 temas de teste sem memória
  python projeto.py teste --versao B         # ... com a memória B congelada
  (o grupo preenche feedback_humano.csv lendo os projetos de treino)
  python projeto.py memoria --fonte humano   # constrói a memória C a partir do feedback humano
  python projeto.py teste --versao C
  python projeto.py avaliacao-cega            # gera os relatórios anonimizados + planilha de notas
  (cada avaliador preenche a sua cópia da planilha)
  python projeto.py analise                   # tabelas, gráficos e testes estatísticos

Decisões de desenho
  - B e C são construídas a partir dos MESMOS projetos de treino e do MESMO extrator de lições;
    a única diferença é o texto do feedback (revisor LLM vs. humano). Isso isola a QP2.
  - Na fase de teste a memória é somente-leitura: os 5 projetos de teste de A, B e C partem
    exatamente do mesmo estado, e a ordem dos temas não interfere.
  - Mesmo modelo e mesma seed de geração por tema nas três versões.

Use --mock em qualquer etapa para testar o fluxo sem LLM (feedback e notas humanas simulados).
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import random
import shutil
import sys

from agent_lab_memory import AgentLaboratory, LLM, MockLLM, LongTermMemory, Embedder
from agent_lab_memory.memory import extract_lessons, format_llm_review, project_facts, store_lessons

VERSOES = {"A": "Sem memória", "B": "Memória + revisor LLM", "C": "Memória + humano"}
CRITERIOS = ["qualidade_experimental", "qualidade_relatorio", "utilidade"]


# ============================================================================ utilidades
def load_topics(path: str) -> tuple[list[str], list[str]]:
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    return d["treino"], d["teste"]


def make_llm(args, seed: int):
    if args.mock:
        return MockLLM(seed=seed)
    return LLM(model=args.model, provider=args.provider, base_url=args.base_url, seed=seed)


def read_csv(path: str) -> list[dict]:
    """Lê CSV separado por ';' ou ',' (Excel em português usa ';', Google Sheets costuma usar ',')."""
    with open(path, encoding="utf-8-sig", newline="") as f:
        sample = f.read(4096)
        f.seek(0)
        try:
            delim = csv.Sniffer().sniff(sample, delimiters=";,").delimiter
        except csv.Error:
            delim = ";"
        return list(csv.DictReader(f, delimiter=delim))


def to_md(df) -> str:
    try:
        return df.to_markdown()
    except ImportError:  # 'tabulate' ausente
        return "```\n" + df.to_string() + "\n```"


def write_csv(path: str, rows: list[dict], fields: list[str]):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, delimiter=";")
        w.writeheader()
        w.writerows(rows)


def project_dirs(base: str, seeds: int, n: int):
    for s in range(seeds):
        for i in range(n):
            yield s, i, os.path.join(base, f"seed{s}", f"t{i}")


def load_result(d: str) -> dict | None:
    p = os.path.join(d, "result.json")
    if not os.path.exists(p):
        return None
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def run_projects(args, topics: list[str], base: str, memory_for_seed, seed_offset: int):
    for s in range(args.seeds):
        memory = memory_for_seed(s)
        for i, topic in enumerate(topics):
            d = os.path.join(base, f"seed{s}", f"t{i}")
            if load_result(d) is not None:
                print(f"[pulando] {d} (já concluído)")
                continue
            print(f"\n[rodando] {d}: {topic[:70]}", flush=True)
            lab = AgentLaboratory(topic, make_llm(args, seed_offset + s * 100 + i), memory=memory,
                                  num_papers=args.num_papers, max_steps=args.max_steps, mle_steps=args.mle_steps,
                                  out_dir=d, verbose=args.verbose)
            r = lab.run()
            u = r["usage"]
            print(f"[ok] nota LLM={r['overall']} | falhas={len(r['failures'])} | "
                  f"tokens={u['prompt_tokens'] + u['completion_tokens']} | {r['elapsed_s']}s", flush=True)


# ============================================================================ etapas
FEEDBACK_FIELDS = ["projeto", "tema", "arquivo_para_ler", "nota_1a5", "o_que_deu_errado", "como_evitar"]


def cmd_treino(args):
    treino, _ = load_topics(args.topics)
    run_projects(args, treino, os.path.join(args.out, "treino"), lambda s: None, seed_offset=0)
    # planilha de feedback humano (condição C): preserva o que já foi preenchido
    path = os.path.join(args.out, "feedback_humano.csv")
    old = {r["projeto"]: r for r in read_csv(path)} if os.path.exists(path) else {}
    rows = []
    for s, i, d in project_dirs(os.path.join(args.out, "treino"), args.seeds, len(treino)):
        pid = f"seed{s}/t{i}"
        rows.append(old.get(pid) or {"projeto": pid, "tema": treino[i],
                                     "arquivo_para_ler": os.path.relpath(os.path.join(d, "resumo_para_feedback.md"), args.out),
                                     "nota_1a5": "", "o_que_deu_errado": "", "como_evitar": ""})
    write_csv(path, rows, FEEDBACK_FIELDS)
    print(f"\nPlanilha de feedback humano: {path}")


def cmd_memoria(args):
    treino, _ = load_topics(args.topics)
    letra = {"llm": "B", "humano": "C"}[args.fonte]
    fonte = {"llm": "llm", "humano": "human"}[args.fonte]
    fb_rows = {}
    if args.fonte == "humano":
        path = os.path.join(args.out, "feedback_humano.csv")
        if not os.path.exists(path):
            sys.exit("Rode a etapa 'treino' antes: ela cria feedback_humano.csv.")
        fb_rows = {r["projeto"]: r for r in read_csv(path)}
        faltando = [p for p, r in fb_rows.items() if not (r.get("o_que_deu_errado", "").strip() or r.get("como_evitar", "").strip())]
        if faltando and not args.mock:
            sys.exit(f"Feedback humano vazio para: {', '.join(faltando)}. Preencha feedback_humano.csv e rode de novo.")
    os.makedirs(os.path.join(args.out, "memoria"), exist_ok=True)
    for s in range(args.seeds):
        mem_path = os.path.join(args.out, "memoria", f"{letra}_seed{s}.json")
        if os.path.exists(mem_path):
            os.remove(mem_path)  # reconstrói do zero (a etapa é determinística dado o feedback)
        mem = LongTermMemory(mem_path, embedder=Embedder(use_dense=not args.mock))
        mem.start_run()
        llm = make_llm(args, 777_000 + s)  # mesma seed para B e C: o extrator se comporta igual
        listagem = [f"# Memória {letra} ({VERSOES[letra]}) · seed {s}\n"]
        for i, topic in enumerate(treino):
            d = os.path.join(args.out, "treino", f"seed{s}", f"t{i}")
            res = load_result(d)
            if res is None:
                sys.exit(f"Projeto de treino ausente: {d}. Rode a etapa 'treino' antes.")
            pid = f"seed{s}/t{i}"
            if fonte == "llm":
                feedback = format_llm_review(res.get("review"))
            else:
                r = fb_rows.get(pid, {})
                if args.mock and not (r.get("o_que_deu_errado") or r.get("como_evitar")):
                    r = {"nota_1a5": "2", "o_que_deu_errado": "O experimento usou poucos exemplos e não reportou "
                         "intervalo de confiança.", "como_evitar": "Usar ao menos 200 exemplos e bootstrap."}
                feedback = (f"Score given by the humans (1-5): {r.get('nota_1a5', '?')}\n"
                            f"What went wrong: {r.get('o_que_deu_errado', '')}\nHow to avoid it: {r.get('como_evitar', '')}")
            lessons = extract_lessons(llm, topic, project_facts(res), feedback, fonte)
            n = store_lessons(mem, lessons, topic, fonte, pid)
            listagem.append(f"\n## {pid} · {topic}\n" + "\n".join(
                f"- **{l['phase']}** · {l['problem']} → {l['how_to_avoid']} (importância {l['importance']:.1f})"
                for l in lessons) + (f"\n_({n} novas; o resto foi fundido com lições parecidas)_" if n < len(lessons) else ""))
        mem.save()
        with open(os.path.join(args.out, "memoria", f"licoes_{letra}_seed{s}.md"), "w", encoding="utf-8") as f:
            f.write("\n".join(listagem) + "\n")
        print(f"Memória {letra} seed {s}: {mem.stats()['insight']} lições -> {mem_path}")


def cmd_teste(args):
    _, teste = load_topics(args.topics)
    v = args.versao

    def memory_for_seed(s):
        if v == "A":
            return None
        p = os.path.join(args.out, "memoria", f"{v}_seed{s}.json")
        if not os.path.exists(p):
            fonte = "llm" if v == "B" else "humano"
            sys.exit(f"Memória {v} não encontrada ({p}). Rode: python projeto.py memoria --fonte {fonte}")
        return LongTermMemory(p, embedder=Embedder(use_dense=not args.mock), read_only=True)

    # mesma seed de geração por (seed, tema) em A, B e C
    run_projects(args, teste, os.path.join(args.out, "teste", v), memory_for_seed, seed_offset=500_000)


def cmd_avaliacao(args):
    _, teste = load_topics(args.topics)
    base = os.path.join(args.out, "avaliacao_cega")
    chave_path = os.path.join(base, ".chave.json")
    if os.path.exists(chave_path) and not args.refazer:
        sys.exit(f"A avaliação cega já foi gerada ({base}). Use --refazer para gerar de novo (embaralha outra vez).")
    itens = []
    for v in VERSOES:
        for s, i, d in project_dirs(os.path.join(args.out, "teste", v), args.seeds, len(teste)):
            if load_result(d) is not None:
                itens.append({"versao": v, "seed": s, "t": i, "dir": d, "tema": teste[i]})
    if not itens:
        sys.exit("Nenhum projeto de teste concluído ainda.")
    random.Random(2026).shuffle(itens)
    shutil.rmtree(base, ignore_errors=True)
    os.makedirs(os.path.join(base, "relatorios"))
    chave, planilha = {}, []
    for n, it in enumerate(itens, 1):
        rid = f"R{n:02d}"
        src = os.path.join(it["dir"], "report.md")
        dst = os.path.join(base, "relatorios", f"{rid}.md")
        if os.path.exists(src) and os.path.getsize(src) > 0:
            shutil.copy(src, dst)
        else:
            with open(dst, "w", encoding="utf-8") as f:
                f.write(f"# {it['tema']}\n\n(O agente não chegou a gerar um relatório neste projeto.)\n")
        chave[rid] = it
        planilha.append({"id": rid, "arquivo": f"relatorios/{rid}.md", "tema": it["tema"],
                         **{c: "" for c in CRITERIOS}, "comentarios": ""})
    with open(chave_path, "w", encoding="utf-8") as f:
        json.dump(chave, f, ensure_ascii=False, indent=2)
    fields = ["id", "arquivo", "tema", *CRITERIOS, "comentarios"]
    write_csv(os.path.join(base, "planilha_modelo.csv"), planilha, fields)
    print(f"{len(itens)} relatórios anonimizados em {base}/relatorios")
    print("Cada avaliador copia planilha_modelo.csv como avaliacao_<seu_nome>.csv (na mesma pasta) e dá notas 1–5.")
    print("NÃO abra .chave.json antes de terminar: ela revela a versão de cada relatório.")
    if args.mock:  # avaliadores simulados, só para testar a análise
        for nome in ("mock1", "mock2"):
            rng = random.Random(nome)
            rows = [{**r, **{c: rng.randint(1, 5) for c in CRITERIOS}} for r in planilha]
            write_csv(os.path.join(base, f"avaliacao_{nome}.csv"), rows, fields)


def cmd_analise(args):
    import pandas as pd
    treino, teste = load_topics(args.topics)
    out = os.path.join(args.out, "analise")
    os.makedirs(out, exist_ok=True)

    # falhas que já ocorreram no treino (a memória "deveria" ter ensinado a evitá-las)
    vistas = set()
    for s, i, d in project_dirs(os.path.join(args.out, "treino"), args.seeds, len(treino)):
        r = load_result(d)
        for f in (r or {}).get("failures", []):
            vistas.add((f["fase"], f["categoria"]))

    rows, falhas = [], []
    for v in VERSOES:
        for s, i, d in project_dirs(os.path.join(args.out, "teste", v), args.seeds, len(teste)):
            r = load_result(d)
            if r is None:
                continue
            fs = r.get("failures", [])
            for f in fs:
                falhas.append({"versao": v, "seed": s, "t": i, "fase": f["fase"], "categoria": f["categoria"],
                               "repetida": (f["fase"], f["categoria"]) in vistas})
            u = r["usage"]
            rows.append({"versao": v, "seed": s, "t": i, "tema": teste[i][:60], "nota_llm": r.get("overall"),
                         "mle_score": r.get("mle_score"), "sucesso_etapas": r.get("workflow_success"),
                         "falhas": len(fs), "falhas_repetidas": sum((f["fase"], f["categoria"]) in vistas for f in fs),
                         "tokens": u["prompt_tokens"] + u["completion_tokens"], "tempo_min": r["elapsed_s"] / 60})
    if not rows:
        sys.exit("Nenhum projeto de teste concluído.")
    df = pd.DataFrame(rows)
    df_f = pd.DataFrame(falhas, columns=["versao", "seed", "t", "fase", "categoria", "repetida"])

    # notas humanas (avaliação cega)
    base = os.path.join(args.out, "avaliacao_cega")
    chave_path = os.path.join(base, ".chave.json")
    tem_humano = False
    if os.path.exists(chave_path):
        chave = json.load(open(chave_path, encoding="utf-8"))
        aval = []
        for fn in sorted(os.listdir(base)):
            if fn.startswith("avaliacao_") and fn.endswith(".csv"):
                for r in read_csv(os.path.join(base, fn)):
                    it = chave.get(r.get("id", "").strip())
                    if not it:
                        continue
                    vals = {c: pd.to_numeric(r.get(c), errors="coerce") for c in CRITERIOS}
                    aval.append({"versao": it["versao"], "seed": it["seed"], "t": it["t"], "avaliador": fn[10:-4], **vals})
        if aval:
            da = pd.DataFrame(aval)
            da["nota_humana"] = da[CRITERIOS].mean(axis=1)
            print(f"Avaliações humanas: {da['avaliador'].nunique()} avaliador(es), {len(da)} notas")
            da.to_csv(os.path.join(out, "avaliacoes_humanas.csv"), index=False)
            media = da.groupby(["versao", "seed", "t"])[CRITERIOS + ["nota_humana"]].mean().reset_index()
            df = df.merge(media, on=["versao", "seed", "t"], how="left")
            tem_humano = True

    df.to_csv(os.path.join(out, "projetos_teste.csv"), index=False)
    def tabela(colunas):  # "média ± desvio", uma linha por versão
        g = df.groupby("versao")[colunas]
        m, d = g.mean(), g.std().fillna(0)
        return m.round(2).astype(str) + " ± " + d.round(2).astype(str)

    qual = (CRITERIOS + ["nota_humana"] if tem_humano else []) + ["nota_llm", "mle_score"]
    proc = ["sucesso_etapas", "falhas", "falhas_repetidas", "tokens", "tempo_min"]
    resumo_q, resumo_p = tabela(qual), tabela(proc)
    por_etapa = (df_f.groupby(["fase", "versao"]).size().unstack(fill_value=0)
                 .reindex(columns=list(VERSOES), fill_value=0)) if len(df_f) else pd.DataFrame()
    n_proj = df.groupby("versao").size()
    if len(por_etapa):
        por_etapa = (por_etapa / n_proj.reindex(por_etapa.columns)).round(2)  # falhas por projeto
    categorias = (df_f.groupby(["categoria", "versao"]).size().unstack(fill_value=0)
                  .reindex(columns=list(VERSOES), fill_value=0)) if len(df_f) else pd.DataFrame()

    md = ["# Resultados (metade de teste)\n", f"Projetos por versão: {n_proj.to_dict()}\n",
          "## Qualidade (média ± desvio)\n", to_md(resumo_q),
          "\n## Falhas, custo e tempo por projeto (média ± desvio)\n", to_md(resumo_p), "\n## Falhas por projeto, por etapa\n",
          to_md(por_etapa) if len(por_etapa) else "(nenhuma falha)",
          "\n## Falhas por categoria (total)\n", to_md(categorias) if len(categorias) else "(nenhuma)"]

    # estatística: comparações pareadas (mesma seed e mesmo tema)
    alvo = "nota_humana" if tem_humano else "nota_llm"
    try:
        from scipy.stats import friedmanchisquare, wilcoxon, spearmanr
        piv = df.pivot_table(index=["seed", "t"], columns="versao", values=alvo).dropna()
        md.append(f"\n## Testes estatísticos ({alvo}, {len(piv)} pares)\n")
        if len(piv) >= 3 and set(VERSOES) <= set(piv.columns):
            st, p = friedmanchisquare(piv["A"], piv["B"], piv["C"])
            md.append(f"- Friedman A×B×C: estatística={st:.2f}, p={p:.3f}")
        for x, y in (("B", "A"), ("C", "A"), ("C", "B")):
            if {x, y} <= set(piv.columns) and len(piv) >= 5 and (piv[x] != piv[y]).any():
                st, p = wilcoxon(piv[x], piv[y])
                md.append(f"- Wilcoxon {x} vs {y}: diferença média={(piv[x] - piv[y]).mean():+.2f}, p={p:.3f}")
        if tem_humano:
            ok = df[["nota_llm", "nota_humana"]].dropna()
            if len(ok) >= 5:
                rho, p = spearmanr(ok["nota_llm"], ok["nota_humana"])
                md.append(f"- Revisor LLM vs. humanos (Spearman): rho={rho:.2f}, p={p:.3f}; "
                          f"média LLM={ok['nota_llm'].mean():.2f}/10 vs. humana={ok['nota_humana'].mean():.2f}/5")
        md.append("\nCom 5 temas por seed o poder estatístico é baixo: trate os p-valores como indicativos.")
    except ImportError:
        md.append("\n(scipy não instalado: testes estatísticos não executados)")

    with open(os.path.join(out, "resumo.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(md) + "\n")
    print("\n".join(md))
    grafico(df, df_f, tem_humano, os.path.join(out, "grafico_resultados.png"))
    print(f"\nArquivos em {out}")


def grafico(df, df_f, tem_humano, path):
    import matplotlib
    import matplotlib.pyplot as plt
    if "ipykernel" not in sys.modules:
        matplotlib.use("Agg")
    cores = {"A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a"}  # paleta categórica validada, ordem fixa
    vs = [v for v in VERSOES if v in set(df["versao"])]
    paineis = [("nota_humana" if tem_humano else "nota_llm",
                "Nota humana (1–5)" if tem_humano else "Nota do revisor LLM (1–10)"),
               ("falhas_repetidas", "Falhas repetidas do treino, por projeto"),
               ("tokens", "Tokens por projeto")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (col, titulo) in zip(axes, paineis):
        g = df.groupby("versao")[col]
        m, e = g.mean().reindex(vs).fillna(0), g.std().reindex(vs).fillna(0)
        x = list(range(len(vs)))
        baixo = [min(e[v], m[v]) for v in vs]  # métricas não negativas: a barra de erro não passa de zero
        ax.bar(x, m, yerr=[baixo, list(e)], color=[cores[v] for v in vs], width=0.6, capsize=4,
               edgecolor="white", linewidth=2, error_kw={"elinewidth": 1, "ecolor": "#555"})
        if col == "nota_humana":
            ax.set_ylim(0, 5)
        else:
            ax.set_ylim(bottom=0, top=max(1, float((m + e).max()), float(df[col].max())) * 1.15)
        topo = ax.get_ylim()[1]
        for xi, v in zip(x, vs):  # pontos individuais: com poucos projetos, a média sozinha engana
            pts = df.loc[df["versao"] == v, col].dropna()
            ax.scatter([xi] * len(pts), pts, color="#333", s=14, zorder=3, alpha=0.6)
            txt = f"{m[v]:.2f}" if col != "tokens" else f"{m[v]/1000:.0f}k"
            dentro = m[v] > 0.12 * topo
            ax.annotate(txt, (xi, 0 if dentro else m[v]), xytext=(0, 4), textcoords="offset points", ha="center",
                        fontsize=9, color="white" if dentro else "#333", fontweight="bold")
        ax.set_xticks(list(x))
        ax.set_xticklabels([f"{v}\n{VERSOES[v]}" for v in vs], fontsize=9)
        ax.set_title(titulo, loc="left", fontsize=11)
        ax.grid(axis="y", color="#e5e5e5")
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    if "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)


# ============================================================================ CLI
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("etapa", choices=["treino", "memoria", "teste", "avaliacao-cega", "analise"])
    ap.add_argument("--fonte", choices=["llm", "humano"], help="etapa memoria: origem das lições")
    ap.add_argument("--versao", choices=list(VERSOES), help="etapa teste: A, B ou C")
    ap.add_argument("--topics", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "topics.json"))
    ap.add_argument("--out", default="resultados_projeto")
    ap.add_argument("--seeds", type=int, default=1)
    ap.add_argument("--model", default="gpt-4o-mini")
    ap.add_argument("--provider", default="openai", choices=["openai", "anthropic"])
    ap.add_argument("--base_url", default=None)
    ap.add_argument("--num_papers", type=int, default=5)
    ap.add_argument("--max_steps", type=int, default=20)
    ap.add_argument("--mle_steps", type=int, default=3)
    ap.add_argument("--mock", action="store_true", help="LLM, feedback e notas simulados (teste do fluxo)")
    ap.add_argument("--refazer", action="store_true", help="avaliacao-cega: gerar de novo")
    ap.add_argument("--verbose", action="store_true")
    args = ap.parse_args()

    if args.etapa == "memoria" and not args.fonte:
        ap.error("informe --fonte llm ou --fonte humano")
    if args.etapa == "teste" and not args.versao:
        ap.error("informe --versao A, B ou C")
    os.makedirs(args.out, exist_ok=True)
    # o código gerado pelo agente usa o mesmo modelo quando o experimento precisa de um LLM
    os.environ["AGENTLAB_LLM_MODEL"] = args.model
    if args.base_url:
        os.environ["AGENTLAB_LLM_BASE_URL"] = args.base_url
    {"treino": cmd_treino, "memoria": cmd_memoria, "teste": cmd_teste,
     "avaliacao-cega": cmd_avaliacao, "analise": cmd_analise}[args.etapa](args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/agent_lab/topics.json
{
  "_nota": "10 temas pequenos: os 5 do artigo + 5 novos. A metade 'treino' forma a memória; a metade 'teste' compara A, B e C. Os temas misturam avaliação de LLMs, visão e ML clássico nas duas metades, para que as lições possam transferir.",
  "treino": [
    "Do language models exhibit cognitive biases, such as confirmation bias or anchoring bias?",
    "Do language models improve accuracy on MedQA when asked to perform differential diagnosis?",
    "Does gender role play affect the accuracy of language models on answering math questions?",
    "Does chain-of-thought prompting improve the accuracy of a small language model on grade-school math (GSM8K)?",
    "Does label noise hurt logistic regression more than gradient boosting on a small tabular classification dataset?"
  ],
  "teste": [
    "Are image transformers more or less sensitive to pixel noise than convolutional networks?",
    "Are language models sensitive to word order in multiple choice benchmarks?",
    "Are sentiment classifiers sensitive to negation words on short movie reviews (SST-2)?",
    "Does rotation data augmentation make a small CNN more robust to rotated images on CIFAR-10?",
    "Does the order of few-shot examples change the accuracy of a language model on text classification?"
  ],
  "_do_artigo": "treino 1-3 e teste 1-2 são os 5 temas da Seção 4.1 do artigo; os demais são novos"
}

## 0.7 Testes rápidos
Fluxo completo com LLM simulado (segundos), resposta do modelo real e conexão com o arXiv.

In [ ]:
!cd /content/agent_lab && rm -rf /content/teste_mock && \
 for e in "treino" "memoria --fonte llm" "teste --versao A" "teste --versao B"; do \
   python projeto.py $$e --mock --out /content/teste_mock --max_steps 6 --num_papers 2 > /dev/null && echo "ok: $$e"; done

import sys
sys.path.insert(0, "/content/agent_lab")
from agent_lab_memory import LLM
from agent_lab_memory.llm import extract_command
from agent_lab_memory.tools import ArxivSearch
llm = LLM(model=MODELO, base_url=BASE_URL, max_tokens=512)
t0 = time.time()
resp = llm.query("You are a postdoc. Reply with exactly one command: ```PLAN\n<plan>\n```",
                 "Write a 2-step plan to compare logistic regression and an MLP on a small dataset.")
print(resp[:500], f"\n--- {time.time() - t0:.1f}s | formato ok: {extract_command(resp, 'PLAN') is not None}")
res = ArxivSearch(max_results=3).summaries("large language models medical question answering")
print("arXiv:", "OK" if "arXiv ID" in res else "PROBLEMA -> " + res[:200])

# Parte 1 · Treino, memória B e testes A e B (até 23/10)
## 1.1 Treino
Roda os 5 temas de treino **uma vez**, sem memória. É a etapa mais longa junto com os testes: anote quanto o
primeiro projeto leva. Se a sessão cair, rode a Parte 0 e esta célula de novo (os concluídos são pulados).

In [ ]:
!cd /content/agent_lab && python projeto.py treino {ARGS} --verbose

## 1.2 Memória B (lições do revisor LLM)
Rápido: uma chamada ao LLM por projeto de treino. As lições ficam legíveis em `memoria/licoes_B_seed0.md`.

In [ ]:
!cd /content/agent_lab && python projeto.py memoria --fonte llm {ARGS}
print(open(f"{PASTA_DRIVE}/memoria/licoes_B_seed0.md", encoding="utf-8").read())

## 1.3 Teste da versão A (sem memória)

In [ ]:
!cd /content/agent_lab && python projeto.py teste --versao A {ARGS} --verbose

## 1.4 Teste da versão B (memória do revisor LLM, congelada)

In [ ]:
!cd /content/agent_lab && python projeto.py teste --versao B {ARGS} --verbose

## 1.5 Resultado parcial (A vs. B, só com métricas automáticas)
A nota usada aqui é a do revisor LLM, que o artigo mostra ser otimista. A comparação principal usa a avaliação humana da Parte 3.

In [ ]:
!cd /content/agent_lab && python projeto.py analise {ARGS}

### Se aparecer `FALHOU` em alguma fase
O log mostra passos, comandos inválidos e tempo de cada fase. A célula abaixo mostra as últimas respostas brutas
do modelo no projeto mais recente.
- **Muitos comandos inválidos:** o modelo não segue o formato. Troque `MODELO` por `qwen2.5:14b`.
- **0 comandos inválidos, mas falhou:** faltaram passos. Aumente `MAX_STEPS` ou use um modelo maior.

Falhas também são **dado do experimento**: elas entram na métrica "falhas que se repetem". Só apague resultados
(`!rm -rf {PASTA_DRIVE}/teste`) se mudar uma configuração (modelo, passos), porque A, B e C precisam usar as mesmas.

In [ ]:
import glob, json
logs = sorted(glob.glob(f"{PASTA_DRIVE}/**/llm_calls.jsonl", recursive=True), key=os.path.getmtime)
if logs:
    print("Projeto:", logs[-1], "\n")
    for c in [json.loads(l) for l in open(logs[-1], encoding="utf-8")][-6:]:
        print("SYSTEM:", c["system"][:120].replace("\n", " ")); print("RESPOSTA:", c["response"][:500]); print("-" * 80)
else:
    print("Nenhum log ainda.")

# Parte 2 · Feedback humano, memória C e teste C (até 13/11)
## 2.1 Feedback humano sobre os projetos de treino
Para cada projeto de treino, o grupo lê o arquivo `resumo_para_feedback.md` (etapas, falhas detectadas, plano,
código, saída e relatório) e responde:
- **nota (1–5)** para o projeto como um todo;
- **o que deu errado**;
- **como evitar** nos próximos projetos.

Pode escrever em português: o extrator de lições traduz. Use o mesmo cuidado que o revisor LLM teria, para que a
comparação B × C seja justa.

**Duas formas de preencher (escolham uma):**
1. **No notebook:** rode a célula abaixo. Ela mostra cada projeto ainda sem feedback e salva a cada resposta.
2. **Na planilha:** baixe `feedback_humano.csv` do Drive, preencha no Excel ou no Google Sheets, salve como CSV e
   suba de volta com o mesmo nome.

In [ ]:
import os, sys
from IPython.display import Markdown, display
sys.path.insert(0, "/content/agent_lab")
from projeto import read_csv, write_csv, FEEDBACK_FIELDS

path = f"{PASTA_DRIVE}/feedback_humano.csv"
rows = read_csv(path)
pendentes = [r for r in rows if not (r["o_que_deu_errado"].strip() or r["como_evitar"].strip())]
print(f"{len(pendentes)} de {len(rows)} projetos sem feedback.")
for r in pendentes:
    resumo = open(os.path.join(PASTA_DRIVE, r["arquivo_para_ler"]), encoding="utf-8").read()
    display(Markdown(f"---\n## {r['projeto']}\n" + resumo[:12000]))
    nota = input(f"[{r['projeto']}] Nota 1-5 (Enter para parar aqui): ").strip()
    if not nota:
        break
    r["nota_1a5"] = nota
    r["o_que_deu_errado"] = input("O que deu errado? ").strip()
    r["como_evitar"] = input("Como evitar nos próximos projetos? ").strip()
    write_csv(path, rows, FEEDBACK_FIELDS)   # salva a cada projeto
    print("salvo.")

## 2.2 Memória C (lições do feedback humano)
Usa os **mesmos projetos** e o **mesmo extrator** da memória B; só muda a fonte do feedback.

In [ ]:
!cd /content/agent_lab && python projeto.py memoria --fonte humano {ARGS}
print(open(f"{PASTA_DRIVE}/memoria/licoes_C_seed0.md", encoding="utf-8").read())

## 2.3 Teste da versão C (memória humana, congelada)

In [ ]:
!cd /content/agent_lab && python projeto.py teste --versao C {ARGS} --verbose

# Parte 3 · Avaliação cega e análise (até 13/11)
## 3.1 Gerar os relatórios anonimizados
Copia os relatórios de teste de A, B e C para `avaliacao_cega/relatorios/R01.md, R02.md...` em ordem aleatória.
**Não abram `.chave.json`** antes de terminar as notas: ela revela a versão de cada relatório.

In [ ]:
!cd /content/agent_lab && python projeto.py avaliacao-cega {ARGS}

## 3.2 Avaliação humana (cada integrante, separadamente)
Critérios do artigo (Seção 4.1), notas de 1 a 5:
- **qualidade_experimental:** qualidade dos resultados experimentais apresentados;
- **qualidade_relatorio:** qualidade da escrita do relatório;
- **utilidade:** quão útil seria um assistente que gera esse relatório como ponto de partida para a pesquisa.

Rode a célula abaixo, escreva seu nome e dê as notas. Cada avaliador gera o próprio arquivo
`avaliacao_<nome>.csv`. Também dá para preencher `planilha_modelo.csv` fora do notebook e salvá-la com esse nome.

In [ ]:
import os, sys
from IPython.display import Markdown, display
sys.path.insert(0, "/content/agent_lab")
from projeto import CRITERIOS, read_csv, write_csv
base = f"{PASTA_DRIVE}/avaliacao_cega"
nome = input("Seu nome (sem espaços): ").strip().lower()
dest = f"{base}/avaliacao_{nome}.csv"
rows = read_csv(dest if os.path.exists(dest) else f"{base}/planilha_modelo.csv")
campos = ["id", "arquivo", "tema", *CRITERIOS, "comentarios"]
for r in rows:
    if all(str(r[c]).strip() for c in CRITERIOS):
        continue
    display(Markdown(f"---\n# {r['id']}\n" + open(os.path.join(base, r["arquivo"]), encoding="utf-8").read()[:15000]))
    q = input(f"[{r['id']}] qualidade_experimental 1-5 (Enter para parar): ").strip()
    if not q:
        break
    r["qualidade_experimental"] = q
    r["qualidade_relatorio"] = input("qualidade_relatorio 1-5: ").strip()
    r["utilidade"] = input("utilidade 1-5: ").strip()
    r["comentarios"] = input("comentários (opcional): ").strip()
    write_csv(dest, rows, campos)
print("Notas salvas em", dest)

## 3.3 Análise final
Junta as notas de todos os avaliadores com a chave e gera as tabelas, o gráfico e os testes estatísticos (Friedman A×B×C e Wilcoxon pareado). Tudo fica salvo em `analise/` no Drive.

In [ ]:
!cd /content/agent_lab && python projeto.py analise {ARGS}
from IPython.display import Image
display(Image(f"{PASTA_DRIVE}/analise/grafico_resultados.png"))

## Onde estão os arquivos (no Drive, em `PASTA_DRIVE`)
| Caminho | Conteúdo |
|---|---|
| `treino/seed0/t*/` | projetos de treino: `resumo_para_feedback.md`, `report.md`, código, `result.json`, `llm_calls.jsonl` |
| `feedback_humano.csv` | o feedback do grupo (entrada da memória C) |
| `memoria/B_seed0.json`, `C_seed0.json` | as memórias; `licoes_*.md` mostra as lições em texto |
| `teste/{A,B,C}/seed0/t*/` | projetos de teste de cada versão |
| `avaliacao_cega/` | relatórios anonimizados, planilhas dos avaliadores e a chave |
| `analise/` | `resumo.md`, `projetos_teste.csv`, `avaliacoes_humanas.csv`, `grafico_resultados.png` |